# A-nanTa V8 — Full-Power 3-Scale Dollar-Bar Ensemble (`250K` / `500K` / `1500K`)

Standalone Full-Power training notebook (separate checkpoints from `A-nanTa_V7314_TRAINING_QUALITY_GATES.ipynb`):
- Select `SCALE_PROFILE` in Cell 2 (`'250K'`, `'500K'`, or `'1500K'`) to train each scale into its own isolated checkpoint folder.
- Full-Power settings for A100/L4: `batch_size=256`, `max_train_steps_per_epoch=5000` (1.28M samples/epoch), full 100% validation (`stage_max_val_steps=None`), extended epochs with Early Stopping (`patience=5`).
- Includes Cell 16 for **3-Scale Ensemble + Non-Overlapping Backtest** (after all 3 models finish).

In [ ]:
# Cell 1 — dependencies and reproducibility
import os, json, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
try:
    import pyarrow  # parquet reader
except ImportError:
    import sys, subprocess
    if 'google.colab' not in sys.modules:
        raise ImportError('pyarrow is required: pip install pyarrow')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
    import pyarrow
import torch
from torch import nn
from torch.nn import functional as F

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True
print('Torch:', torch.__version__, 'device:', DEVICE)
USE_BF16 = (DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported())
from contextlib import nullcontext
def amp_context():
    return torch.autocast(device_type='cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()
print('BF16 autocast enabled:', USE_BF16)

In [ ]:
# Cell 2 — Select scale profile: '250K', '500K', or '1500K'
SCALE_PROFILE = '500K'  # Change to '250K' or '1500K' for the other 2 ensemble members
_SCALES = {
    '100K':  (100000.0,  'BTCUSD_DOLLAR_100K.parquet',  '/content/manticore_v8_btc_dollar_100k_cache',  'v8_fullpower_100k_checkpoints'),
    '250K':  (250000.0,  'BTCUSD_DOLLAR_250K.parquet',  '/content/manticore_v8_btc_dollar_250k_cache',  'v8_fullpower_250k_checkpoints'),
    '500K':  (500000.0,  'BTCUSD_DOLLAR_500K.parquet',  '/content/manticore_v8_btc_dollar_500k_cache',  'v8_fullpower_500k_checkpoints'),
    '1500K': (1500000.0, 'BTCUSD_DOLLAR_1500K.parquet', '/content/manticore_v8_btc_dollar_1500k_cache',  'v8_fullpower_1500k_checkpoints'),
    '3000K': (3000000.0, 'BTCUSD_DOLLAR_3000K.parquet', '/content/manticore_v8_btc_dollar_3000k_cache', 'v8_fullpower_3000k_checkpoints'),
}
assert SCALE_PROFILE in _SCALES, f"Unknown SCALE_PROFILE: {SCALE_PROFILE}"
_TARGET_USD, _PARQUET_NAME, _CACHE_DIR, _OUT_SUBDIR = _SCALES[SCALE_PROFILE]
print(f"Active Full-Power Profile: {SCALE_PROFILE} (${_TARGET_USD:,.0f}/bar) -> {_OUT_SUBDIR}")

CFG = {
    'data_dir': '/content/manticore_v730_crypto_m5',
    'use_google_drive': True,
    'assets': ['BTCUSD'],
    'pool_assets': ['BTCUSD'],
    'target_asset': 'BTCUSD',
    'context_assets': [],
    'cache_dir': _CACHE_DIR,
    'output_dir': f'/content/drive/MyDrive/HERMES_Colab/ai_training/manticore_v70_trend_specialist_v2/{_OUT_SUBDIR}',
    'timestamp_semantics': 'close',  # Dollar bar timestamps are bar CLOSE times
    'horizons': [2, 6, 16, 48],      # Event-bar horizons (dollar bars, tuned from V7314 diagnostics)
    'lookback': 256,
    'purge_bars': 256,
    'max_gap_seconds': 3600.0,       # Outage threshold (>1h with zero trades)
    'binance_market': 'futures/cm',
    'binance_symbol': 'BTCUSD_PERP',
    'binance_contract_usd': 100.0,
    'binance_auto_download': True,
    'dollar_bar_target_usd': _TARGET_USD,
    'verify_binance_sha256': True,
    'checkpoint_every_steps': 1000,
    'validation_report_months': 2,
    'data_pipeline_version': 'v8_dollar_btc_only_r2',
    'trailing_feature_window': 200,
    'binance_start': '2020-10',
    'binance_end': None,
    'binance_archive_dir': '/content/drive/MyDrive/HERMES_Colab/ai_training/binance_vision/cm_BTCUSD_PERP_aggTrades',
    'tick_bar_cache': f'/content/drive/MyDrive/HERMES_Colab/ai_training/historical_market_cache/binance_cm_btcusd_aggtrades/{_PARQUET_NAME}',
    'train_years': 5, 'val_months': 6, 'cal_months': 3, 'sealed_months': 3,
    'recent_half_life_days': 270., 'recent_uniform_floor': 0.40,
    'stress_boost': 1.25, 'stress_vol_quantile': 0.90, 'stress_cap': 3.0,
    'batch_size': 1024, 'lr': 3.0e-4, 'weight_decay': 0.06, 'dropout': 0.28, 'label_smoothing': 0.03,
    'path_rank': 4, 'path_knots': [1, 2, 4, 6, 8, 12, 16, 24, 32, 48],
    'mtf_timeframes': {'M15': '15min', 'H1': '1h', 'H4': '4h', 'D1': '1D'},
    'mtf_window': 32, 'mtf_encoder_layers': 2,
    'hidden': 256, 'transformer_layers': 5, 'attention_heads': 8, 'gru_layers': 2,
    'expert_width': 512, 'experts': 8, 'top_k': 2,
    'loader_workers': 10, 'prefetch_factor': 4, 'persistent_workers': True,
    'use_bf16': True,
    'neutral_move_bps': 5., 'label_vol_multiplier': .25,
    'barrier_vol_multiplier': 1., 'reference_cost_bps': 6.0,
    'min_rows': 2000, 'min_train_sequences': 500,
    'max_train_steps_per_epoch': 2500,
    'direction_weight': 1., 'decision_weight': 1.8, 'quantile_weight': 1.,
    'rv_weight': .5, 'excursion_weight': .5, 'barrier_weight': .5, 'regime_weight': .25,
    'route_balance_weight': .15, 'pattern_weight': .35, 'jump_weight': .25, 'path_weight': .35,
    'stage_epochs': {'A_SSL': 8, 'B_REGIME': 5, 'D_FORECAST': 25, 'E_PATH': 20, 'F_EXPERTS': 20, 'G_DECISION': 12, 'H_JOINT': 12},
    'stage_lr_mult': {'A_SSL': 1., 'B_REGIME': .65, 'D_FORECAST': .35, 'E_PATH': .15, 'F_EXPERTS': .3, 'G_DECISION': .4, 'H_JOINT': .15},
    'stage_patience': {'A_SSL': 4, 'B_REGIME': 3, 'D_FORECAST': 5, 'E_PATH': 5, 'F_EXPERTS': 5, 'G_DECISION': 4, 'H_JOINT': 4},
    'stage_max_val_steps': 400,
    'research_adverse_penalty': 0.20,
    'path_nll_weight': .20, 'action_q_weight': .35, 'pattern_detail_weight': .25,
    'use_pcgrad_joint': True, 'expert_val_batches': 3,
    'validate_expert_ablation': True,
    'diagnostic_gradient_assert': True,
    'run_sealed_test': False,  # explicit one-time review after freeze
}
assert 3 <= CFG['train_years'] <= 5 and CFG['recent_half_life_days'] > 0
assert 0 < CFG['recent_uniform_floor'] < 1 and CFG['stress_boost'] >= 1
assert max(CFG['horizons']) < CFG['purge_bars']
assert CFG['dollar_bar_target_usd'] > 0
assert CFG['assets'] == CFG['pool_assets'] == ['BTCUSD'], 'BTC-only supervised asset contract'
assert CFG['target_asset'] == 'BTCUSD' and CFG['context_assets'] == [], 'No non-BTC input assets'
assert CFG['path_knots'][-1] == max(CFG['horizons']) and sorted(set(CFG['path_knots'])) == CFG['path_knots']
USE_BF16 = bool(USE_BF16 and CFG.get('use_bf16', True))

## 1. Dollar-Bar Event Data + Causal Microstructure Features
Raw `BTCUSD_PERP` `aggTrades` are streamed into `$500,000` Dollar Bars. Each row represents one completed USD-volume event with strictly monotonic UTC close timestamps, `duration_sec`, `signed_notional_usd`, `buy_notional_usd`, and `trade_count`. All rolling transforms are strictly causal and reset across exchange outages (`dt > max_gap_seconds`).


In [ ]:
# Cell 4 — Dollar-bar loader and causal event-time quantitative features
FEATURES = [
    'ret1', 'ret3', 'ret12', 'vol12', 'vol48', 'vol_ratio', 'efficiency', 'range_norm',
    'position', 'downside', 'skew24', 'kurt24', 'entropy24',
    'spread_rel', 'spread_present', 'volume_log', 'volume_present',
    'trend_strength_96', 'volatility_har_12', 'volatility_har_48', 'volatility_har_288',
    'hurst_proxy', 'wavelet_band', 'jump_proxy', 'mean_reversion_z', 'autocorr24',
    'taker_dollar_imbalance', 'taker_buy_ratio', 'notional_log', 'trade_count_log',
    'imbalance_ewm', 'flow_shock',
    'log_dt', 'dollar_speed_log', 'dollar_imbalance_per_sec',
    'vpin_24', 'kyle_lambda_24', 'gk_vol_24', 'amihud_24', 'avg_trade_size_log',
    'candle_body', 'upper_wick', 'lower_wick', 'ofi_z24', 'vwap_dev_24',
    'rsi_14', 'tod_sin', 'tod_cos',
]

def load_asset_file(path):
    path = Path(path)
    d = pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)
    d.columns = [c.strip().lower() for c in d.columns]
    if 'timestamp' not in d.columns:
        for key in ('datetime', 'date_time', 'time', 'date'):
            if key in d.columns:
                d = d.rename(columns={key: 'timestamp'}); break
    if 'timestamp' not in d.columns and isinstance(d.index, pd.DatetimeIndex):
        d = d.reset_index().rename(columns={d.index.name or 'index': 'timestamp'})
    if 'timestamp' in d.columns and pd.api.types.is_numeric_dtype(d['timestamp']):
        raise ValueError(f'{path}: numeric epoch timestamps need explicit unit')
    missing = {'timestamp', 'open', 'high', 'low', 'close'} - set(d.columns)
    if missing:
        raise ValueError(f'{path}: missing {sorted(missing)}')
    d['timestamp'] = pd.to_datetime(d['timestamp'], utc=True, errors='raise')
    if d['timestamp'].isna().any() or d['timestamp'].duplicated().any():
        raise ValueError(f'{path}: null/duplicate timestamps')
    d = d.sort_values('timestamp').set_index('timestamp')
    for k in ('open', 'high', 'low', 'close', 'bid', 'ask', 'volume', 'notional_usd',
              'signed_notional_usd', 'buy_notional_usd', 'trade_count', 'duration_sec'):
        if k in d:
            d[k] = pd.to_numeric(d[k], errors='coerce')
    if d[['open', 'high', 'low', 'close']].isna().any().any():
        raise ValueError(f'{path}: missing prices')
    if len(d) < CFG['min_rows']:
        raise ValueError(f'{path}: insufficient bars {len(d)}')
    if ((d[['open', 'high', 'low', 'close']] <= 0).any().any()
            or (d.high < d[['open', 'close', 'low']].max(axis=1)).any()
            or (d.low > d[['open', 'close', 'high']].min(axis=1)).any()):
        raise ValueError(f'{path}: invalid OHLC')
    return d

def fast_permutation_entropy(log_returns, window=24):
    a = np.asarray(log_returns, dtype=np.float64)
    n = len(a); w = window - 2
    out = np.full(n, np.nan, dtype=np.float32)
    if n < window:
        return out
    x = a[:-2]; y = a[1:-1]; z = a[2:]
    rank_x = ((y < x).astype('int8') + (z < x).astype('int8'))
    rank_y = ((x <= y).astype('int8') + (z < y).astype('int8'))
    rank_z = 3 - rank_x - rank_y
    patterns = np.full(n, -1, dtype=np.int16)
    good = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    patterns[2:][good] = (rank_x[good] * 9 + rank_y[good] * 3 + rank_z[good]).astype(np.int16)
    bad_cs = np.empty(n + 1, dtype=np.int32); bad_cs[0] = 0
    np.cumsum(patterns < 0, out=bad_cs[1:])
    valid = (bad_cs[w:] - bad_cs[:-w]) == 0
    ent = np.zeros(n - w + 1, dtype=np.float64)
    cs = np.empty(n + 1, dtype=np.int32); cs[0] = 0
    for key in (5, 7, 11, 15, 19, 21):
        np.cumsum(patterns == key, out=cs[1:])
        cnt = cs[w:] - cs[:-w]
        nz = cnt > 0
        pr = cnt[nz] / float(w)
        ent[nz] -= pr * np.log(pr)
    out[w - 1:] = np.where(valid, ent / np.log(6.0), np.nan).astype(np.float32)
    return out

def _causal_features_one_contiguous_segment(d):
    p = d.close; lr = np.log(p).diff(); v = lr.rolling(48, min_periods=24).std()
    f = pd.DataFrame(index=d.index)
    f['ret1'] = lr; f['ret3'] = np.log(p / p.shift(3)); f['ret12'] = np.log(p / p.shift(12))
    f['vol12'] = lr.rolling(12, min_periods=12).std(); f['vol48'] = v
    f['vol_ratio'] = f.vol12 / (v + 1e-8)
    f['efficiency'] = p.diff(24).abs() / (p.diff().abs().rolling(24).sum() + 1e-8)
    f['range_norm'] = (d.high - d.low) / (p * v.replace(0, np.nan) + 1e-8)
    f['position'] = (p - p.rolling(24).min()) / (p.rolling(24).max() - p.rolling(24).min() + 1e-8)
    f['downside'] = lr.clip(upper=0).pow(2).rolling(24).mean().pow(.5)
    f['skew24'] = lr.rolling(24).skew(); f['kurt24'] = lr.rolling(24).kurt()
    f['entropy24'] = fast_permutation_entropy(lr.to_numpy(), 24)
    if {'bid', 'ask'} <= set(d.columns):
        raw = ((d.ask - d.bid) / p).where((d.ask >= d.bid) & (d.bid > 0))
        f['spread_rel'] = raw; f['spread_present'] = raw.notna().astype(float)
    else:
        f['spread_rel'] = np.nan; f['spread_present'] = 0.
    if 'volume' in d:
        x = np.log1p(d.volume.clip(lower=0))
        f['volume_log'] = x - x.rolling(48, min_periods=24).mean()
        f['volume_present'] = d.volume.notna().astype(float)
    else:
        f['volume_log'] = np.nan; f['volume_present'] = 0.
    f['trend_strength_96'] = np.log(p / p.shift(96)) / (lr.rolling(96, min_periods=48).std() * np.sqrt(96) + 1e-9)
    f['volatility_har_12'] = lr.pow(2).rolling(12, min_periods=12).mean()
    f['volatility_har_48'] = lr.pow(2).rolling(48, min_periods=48).mean()
    f['volatility_har_288'] = lr.pow(2).rolling(288, min_periods=144).mean()
    f['hurst_proxy'] = np.log((lr.rolling(48, min_periods=24).sum().abs() + 1e-9) / (lr.abs().rolling(48, min_periods=24).sum() + 1e-9))
    f['wavelet_band'] = lr.rolling(8, min_periods=8).mean() - lr.rolling(48, min_periods=24).mean()
    f['jump_proxy'] = (lr.abs() / (lr.abs().rolling(48, min_periods=24).median() + 1e-9)).clip(upper=50)
    f['mean_reversion_z'] = (np.log(p) - np.log(p).rolling(48, min_periods=24).mean()) / (v * np.sqrt(48) + 1e-8)
    f['autocorr24'] = lr.rolling(24, min_periods=24).corr(lr.shift(1))

    notional = d['notional_usd'] if 'notional_usd' in d else pd.Series(np.nan, index=d.index)
    signed = d['signed_notional_usd'] if 'signed_notional_usd' in d else pd.Series(np.nan, index=d.index)
    buys = d['buy_notional_usd'] if 'buy_notional_usd' in d else pd.Series(np.nan, index=d.index)
    trades = d['trade_count'] if 'trade_count' in d else pd.Series(np.nan, index=d.index)
    imbalance = (signed / (notional + 1e-9)).where(notional > 0)
    f['taker_dollar_imbalance'] = imbalance
    f['taker_buy_ratio'] = (buys / (notional + 1e-9)).where(notional > 0)
    f['notional_log'] = np.log1p(notional.clip(lower=0)) - np.log1p(notional.clip(lower=0)).rolling(48, min_periods=24).mean()
    f['trade_count_log'] = np.log1p(trades.clip(lower=0)) - np.log1p(trades.clip(lower=0)).rolling(48, min_periods=24).mean()
    f['imbalance_ewm'] = imbalance.ewm(span=24, adjust=False, min_periods=12).mean()
    f['flow_shock'] = (signed.abs() / (signed.abs().rolling(48, min_periods=24).median() + 1e-9)).clip(0, 50)

    # Event-time features: duration_sec and dollar speed
    if 'duration_sec' in d:
        dt = d['duration_sec'].clip(lower=1e-3)
    else:
        dt = pd.Series(d.index.to_series().diff().dt.total_seconds().fillna(1.0), index=d.index).clip(lower=1e-3)
    f['log_dt'] = np.log1p(dt)
    speed = np.log1p((notional.clip(lower=0) / dt).clip(lower=0))
    f['dollar_speed_log'] = speed - speed.rolling(48, min_periods=24).mean()
    f['dollar_imbalance_per_sec'] = np.arcsinh(signed / (dt * 1000.0 + 1e-3))
    # Quant microstructure factors (VPIN, Kyle's Lambda, Garman-Klass vol, Amihud, avg trade size)
    f['vpin_24'] = (signed.abs() / (notional + 1e-9)).rolling(24, min_periods=12).mean()
    notional_m = (notional / 1e6).clip(lower=1e-6)
    signed_m = (signed.abs() / 1e6).clip(lower=1e-6)
    f['kyle_lambda_24'] = np.log1p((lr.abs() / signed_m).rolling(24, min_periods=12).mean().clip(lower=0))
    hl = np.log(d.high / d.low)
    co = np.log(d.close / d.open)
    gk = (0.5 * hl.pow(2) - (2.0 * np.log(2.0) - 1.0) * co.pow(2)).clip(lower=0)
    f['gk_vol_24'] = gk.rolling(24, min_periods=12).mean().pow(0.5)
    f['amihud_24'] = np.log1p((lr.abs() / notional_m).rolling(24, min_periods=12).mean().clip(lower=0))
    avg_sz = np.log1p((notional.clip(lower=0) / trades.clip(lower=1.0)).clip(lower=0))
    f['avg_trade_size_log'] = avg_sz - avg_sz.rolling(48, min_periods=24).mean()
    bar_span = (d.high - d.low).clip(lower=1e-9)
    f['candle_body'] = (p - d.open) / bar_span
    f['upper_wick'] = (d.high - np.maximum(d.open, p)) / bar_span
    f['lower_wick'] = (np.minimum(d.open, p) - d.low) / bar_span
    f['ofi_z24'] = (imbalance - imbalance.rolling(24, min_periods=12).mean()) / (imbalance.rolling(24, min_periods=12).std() + 1e-8)
    pos_notional = notional.clip(lower=0)
    vwap_den = pos_notional.rolling(24, min_periods=12).sum()
    vwap24 = (p * pos_notional).rolling(24, min_periods=12).sum() / (vwap_den + 1e-9)
    f['vwap_dev_24'] = np.log(p / (vwap24 + 1e-9)).where((pos_notional > 0) & (vwap_den > 0))
    up14 = lr.clip(lower=0).ewm(alpha=1.0 / 14.0, adjust=False, min_periods=14).mean()
    dn14 = (-lr.clip(upper=0)).ewm(alpha=1.0 / 14.0, adjust=False, min_periods=14).mean()
    f['rsi_14'] = up14 / (up14 + dn14 + 1e-9) - 0.5
    tod_frac = (d.index.hour * 3600 + d.index.minute * 60 + d.index.second) / 86400.0
    f['tod_sin'] = np.sin(2.0 * np.pi * tod_frac).astype(np.float32)
    f['tod_cos'] = np.cos(2.0 * np.pi * tod_frac).astype(np.float32)
    return f.replace([np.inf, -np.inf], np.nan)

def causal_features(d, max_gap_seconds=None):
    '''Split contiguous segments only at true exchange outages (> max_gap_seconds).'''
    if d.empty or len(d) < 2:
        return _causal_features_one_contiguous_segment(d)
    times = d.index.to_numpy(dtype='datetime64[ns]')
    steps = np.diff(times).astype('timedelta64[ns]').astype(np.int64)
    if np.any(steps <= 0):
        raise ValueError('Non-monotonic feature source timestamps')
    if max_gap_seconds is None:
        max_gap_seconds = float(CFG['max_gap_seconds'])
    gap_ns = int(max_gap_seconds * 1e9)
    boundaries = np.r_[0, np.where(steps > gap_ns)[0] + 1, len(d)]
    if len(boundaries) == 2:
        return _causal_features_one_contiguous_segment(d)
    parts = [_causal_features_one_contiguous_segment(d.iloc[a:b]) for a, b in zip(boundaries[:-1], boundaries[1:])]
    return pd.concat(parts).reindex(d.index)

In [ ]:
# Cell 5 — Future-only supervised targets on Dollar-Bar events (outage-safe, low-RAM)
def build_labels(d, horizons):
    close_arr = d.close.to_numpy(dtype=np.float64)
    hi = d.high.to_numpy(dtype=np.float64)
    lo = d.low.to_numpy(dtype=np.float64)
    n = len(d)

    lr = np.log(d.close); r = lr.diff(); y = pd.DataFrame(index=d.index)
    local_vol = r.rolling(48, min_periods=24).std().clip(lower=1e-8)
    eff = (d.close.diff(24).abs() / (d.close.diff().abs().rolling(24).sum() + 1e-8)).fillna(0)
    vratio = (r.rolling(12, min_periods=12).std() / (local_vol + 1e-8)).fillna(1)
    y['regime_trend'] = eff.clip(0, 1).astype('float32')
    y['regime_high_vol'] = (vratio > 1.5).astype('float32')
    y['regime_transition'] = ((eff - eff.shift(12)).abs() > .2).astype('float32')

    r_np = r.to_numpy(dtype=np.float64)
    sq_cs = np.empty(n + 1, dtype=np.float64); sq_cs[0] = 0.0
    np.cumsum(np.nan_to_num(r_np * r_np, nan=0.0), out=sq_cs[1:])
    bad_cs = np.empty(n + 1, dtype=np.int32); bad_cs[0] = 0
    np.cumsum(~np.isfinite(r_np), out=bad_cs[1:])

    times = d.index.to_numpy(dtype='datetime64[ns]')
    steps_ns = np.diff(times).astype('timedelta64[ns]').astype(np.int64)
    gap_ns = int(float(CFG['max_gap_seconds']) * 1e9)
    gap_prefix = np.r_[0, np.cumsum((steps_ns > gap_ns).astype(np.int64))]

    def no_gap_span(h):
        ok = np.zeros(n, dtype=bool)
        ok[:-h] = (gap_prefix[h:] - gap_prefix[:-h]) == 0
        return ok

    local_vol_np = local_vol.to_numpy(dtype=np.float64)
    past_high = d.high.rolling(48, min_periods=24).max().shift(1).to_numpy(dtype=np.float64)
    past_low = d.low.rolling(48, min_periods=24).min().shift(1).to_numpy(dtype=np.float64)
    lr_np = lr.to_numpy(dtype=np.float64)

    for h in horizons:
        future = np.full(n, np.nan, dtype=np.float64)
        future[:-h] = lr_np[h:] - lr_np[:-h]
        y[f'ret_{h}'] = future.astype('float32')
        threshold = np.maximum(CFG['neutral_move_bps'] * 1e-4, CFG['label_vol_multiplier'] * local_vol_np * np.sqrt(h))
        dir_arr = np.where(future > threshold, 2.0, np.where(future < -threshold, 0.0, 1.0)).astype('float32')
        dir_arr[~np.isfinite(future)] = np.nan
        y[f'dir_{h}'] = dir_arr

        max_hi = np.full(n, -np.inf, dtype=np.float64)
        min_lo = np.full(n, np.inf, dtype=np.float64)
        valid = np.ones(n, dtype=bool); valid[n - h:] = False
        upper = np.maximum(CFG['barrier_vol_multiplier'] * local_vol_np * np.sqrt(h), CFG['neutral_move_bps'] * 1e-4)
        up = close_arr * np.exp(upper); dn = close_arr * np.exp(-upper)
        barrier = np.ones(n, dtype=np.int8)
        undecided = np.isfinite(upper) & (upper >= CFG['reference_cost_bps'] * 1e-4)
        ambiguous = np.zeros(n, dtype=bool)
        for j in range(1, h + 1):
            m = n - j
            fhi = hi[j:]; flo = lo[j:]
            valid[:m] &= np.isfinite(fhi) & np.isfinite(flo)
            max_hi[:m] = np.maximum(max_hi[:m], fhi)
            min_lo[:m] = np.minimum(min_lo[:m], flo)
            und = undecided[:m]
            if und.any():
                hit_up = (fhi >= up[:m]) & und
                hit_dn = (flo <= dn[:m]) & und
                both = hit_up & hit_dn
                ambiguous[:m] |= both
                barrier[:m][hit_up & ~both] = 2
                barrier[:m][hit_dn & ~both] = 0
                undecided[:m][hit_up | hit_dn] = False

        with np.errstate(divide='ignore', invalid='ignore'):
            max_exc = np.where(valid, np.log(max_hi / close_arr), np.nan)
            min_exc = np.where(valid, np.log(min_lo / close_arr), np.nan)
        y[f'mfe_{h}'] = max_exc.astype('float32')
        y[f'mae_{h}'] = min_exc.astype('float32')

        rv = np.full(n, np.nan, dtype=np.float32)
        ok_rv = (bad_cs[h + 1:] - bad_cs[1:-h]) == 0
        rv[:n - h] = np.where(ok_rv, sq_cs[h + 1:] - sq_cs[1:-h], np.nan).astype('float32')
        y[f'rv_{h}'] = rv

        bar_arr = np.where(ambiguous, np.nan, barrier.astype('float32'))
        bar_arr[n - h:] = np.nan
        y[f'barrier_{h}'] = bar_arr
        y[f'pattern_{h}'] = np.where(np.isfinite(past_high) & np.isfinite(past_low) & valid,
            ((close_arr * np.exp(max_exc) > past_high) | (close_arr * np.exp(min_exc) < past_low)).astype('float32'), np.nan)
        y[f'jump_{h}'] = np.where(valid, ((max_exc - min_exc) > 3.5 * np.maximum(local_vol_np * np.sqrt(h), 1e-8)).astype('float32'), np.nan)
        y[f'path_{h}'] = np.where(valid, np.clip(future / np.maximum(max_exc - min_exc, 1e-8), -2.0, 2.0), np.nan).astype('float32')
        connected = no_gap_span(h)
        for suffix in ('ret', 'dir', 'mfe', 'mae', 'rv', 'barrier', 'pattern', 'jump', 'path'):
            y.loc[~connected, f'{suffix}_{h}'] = np.nan

    y['decision'] = y[f'barrier_{horizons[-1]}']
    past_mom = np.log(d.close / d.close.shift(24)).to_numpy(dtype=np.float64)
    for h in horizons:
        future = y[f'ret_{h}'].to_numpy(dtype=np.float64)
        complete = np.isfinite(future)
        terminal = np.full(n, np.nan, dtype=np.float64); terminal[:-h] = close_arr[h:]
        mfe = y[f'mfe_{h}'].to_numpy(dtype=np.float64)
        mae = y[f'mae_{h}'].to_numpy(dtype=np.float64)
        ok_p = complete & np.isfinite(past_high) & np.isfinite(past_low) & np.isfinite(past_mom)
        touched = ((close_arr * np.exp(mfe) > past_high) | (close_arr * np.exp(mae) < past_low))
        inside = (terminal <= past_high) & (terminal >= past_low)
        y[f'false_breakout_{h}'] = np.where(ok_p, (touched & inside).astype('float32'), np.nan)
        move_cut = CFG['label_vol_multiplier'] * local_vol_np * np.sqrt(h)
        past_signal = np.sign(past_mom)
        real_move = np.abs(future) > np.maximum(move_cut, CFG['neutral_move_bps'] * 1e-4)
        y[f'reversal_{h}'] = np.where(ok_p, (real_move & (future * past_signal < 0)).astype('float32'), np.nan)
        y[f'continuation_{h}'] = np.where(ok_p, (real_move & (future * past_signal > 0)).astype('float32'), np.nan)
        connected = no_gap_span(h)
        for suffix in ('false_breakout', 'reversal', 'continuation'):
            y.loc[~connected, f'{suffix}_{h}'] = np.nan

    previous = 0
    for k in CFG['path_knots']:
        inc = np.full(n, np.nan, dtype=np.float32)
        inc[:-k] = np.log(close_arr[k:] / close_arr[previous:n - k + previous]).astype('float32')
        inc[~no_gap_span(k)] = np.nan
        y[f'path_increment_{k}'] = inc
        previous = k
    return y

## 2. Binance Vision `BTCUSD_PERP` `aggTrades` → True Dollar Bars
Streams verified monthly `aggTrades` ZIP archives into `$500,000` Dollar Bars (each bar closes when cumulative USD notional reaches `$500,000`). Multiple bars closing within the same millisecond receive deterministic sub-millisecond nanosecond offsets (`+1 ns`) so timestamps remain strictly monotonic and unique.


In [ ]:
# Cell 5.5 — Verified Binance Vision COIN-M BTCUSD_PERP aggTrades -> True Dollar Bars
import io, zipfile, urllib.request, urllib.error, gc

if CFG['use_google_drive'] and not Path('/content/drive/MyDrive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

def _month_range(start_ym, end_ym):
    return [str(p) for p in pd.period_range(start_ym, end_ym, freq='M')]

def _sha256_file(path, chunk=2**20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def _verify_binance_zip_sha256(zpath, base_url):
    marker = zpath.with_suffix(zpath.suffix + '.sha256_verified.json')
    stat = zpath.stat()
    if marker.exists():
        try:
            saved = json.loads(marker.read_text())
            if saved.get('size') == stat.st_size and saved.get('mtime_ns') == stat.st_mtime_ns:
                return saved['expected_sha256']
        except (OSError, ValueError, KeyError):
            pass
    csum_url = f'{base_url}.CHECKSUM'
    csum_path = zpath.with_suffix('.zip.CHECKSUM')
    if not csum_path.exists():
        urllib.request.urlretrieve(csum_url, csum_path)
    expected = csum_path.read_text().strip().split()[0].lower()
    if len(expected) != 64 or any(c not in '0123456789abcdef' for c in expected):
        raise ValueError(f'Invalid CHECKSUM format in {csum_path}')
    actual = _sha256_file(zpath)
    if actual != expected:
        raise ValueError(f'SHA-256 mismatch for {zpath.name}: {actual} != {expected}')
    marker.write_text(json.dumps({
        'file': zpath.name, 'expected_sha256': expected, 'actual_sha256': actual,
        'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns
    }, indent=2))
    return expected

def fetch_binance_month(ym, sym='BTCUSD_PERP', market='futures/cm', archive=CFG['binance_archive_dir']):
    archive = Path(archive); archive.mkdir(parents=True, exist_ok=True)
    name = f'{sym}-aggTrades-{ym}.zip'
    zpath = archive / name
    base_url = f'https://data.binance.vision/data/{market}/monthly/aggTrades/{sym}/{name}'
    if not zpath.exists():
        if not CFG['binance_auto_download']:
            raise FileNotFoundError(zpath)
        part = zpath.with_suffix('.zip.part')
        try:
            urllib.request.urlretrieve(base_url, part)
            part.replace(zpath)
        except urllib.error.HTTPError as exc:
            part.unlink(missing_ok=True)
            if exc.code == 404:
                raise FileNotFoundError(base_url) from exc
            raise
    if CFG['verify_binance_sha256']:
        _verify_binance_zip_sha256(zpath, base_url)
    return zpath

def _read_aggtrade_chunks(zpath, chunksize=2_000_000):
    cols = ['agg_trade_id', 'price', 'quantity', 'first_trade_id', 'last_trade_id', 'transact_time', 'is_buyer_maker']
    with zipfile.ZipFile(zpath) as zf:
        csv_name = [n for n in zf.namelist() if n.endswith('.csv')][0]
        with zf.open(csv_name) as fh:
            head = fh.readline().decode('utf-8', 'ignore').lower()
        has_header = 'price' in head or 'transact_time' in head
        with zf.open(csv_name) as fh:
            yield from pd.read_csv(
                fh, header=0 if has_header else None,
                names=None if has_header else cols,
                chunksize=chunksize,
                usecols=['agg_trade_id', 'price', 'quantity', 'transact_time', 'is_buyer_maker'] if has_header else [0, 1, 2, 5, 6]
            )

def convert_binance_aggtrades_to_dollar_bars(files, outfile, market='futures/cm', asset='BTCUSD'):
    threshold = float(CFG['dollar_bar_target_usd'])
    contract_usd = float(CFG['binance_contract_usd'])
    bars = []
    total = 0; id_disorder_count = 0; id_disorders = []; last_seen_id = None
    last_ts_ns = -1

    # Running bar state carried seamlessly across chunks and monthly files
    cur_open = np.nan; cur_high = -np.inf; cur_low = np.inf
    cur_open_ts = -1
    cur_overshoot = 0.0; cur_usd = 0.0; cur_signed = 0.0; cur_buys = 0.0; cur_trades = 0

    for path in files:
        monthly = 0
        for chunk in _read_aggtrade_chunks(path):
            chunk.columns = [c.strip().lower() for c in chunk.columns]
            if 'agg_trade_id' in chunk.columns:
                ids = pd.to_numeric(chunk['agg_trade_id'], errors='coerce').to_numpy(dtype='int64')
                if len(ids):
                    if last_seen_id is not None and ids[0] <= last_seen_id:
                        id_disorder_count += 1
                        if len(id_disorders) < 10:
                            id_disorders.append({'file': path.name, 'prev_id': int(last_seen_id), 'next_id': int(ids[0])})
                    bad_step = np.where(np.diff(ids) <= 0)[0]
                    if len(bad_step):
                        id_disorder_count += int(len(bad_step))
                        if len(id_disorders) < 10:
                            for idx_b in bad_step[:10 - len(id_disorders)]:
                                id_disorders.append({'file': path.name, 'prev_id': int(ids[idx_b]), 'next_id': int(ids[idx_b + 1])})
                    last_seen_id = int(ids[-1])
            q = pd.DataFrame({
                'price': pd.to_numeric(chunk['price'], errors='coerce'),
                'qty': pd.to_numeric(chunk['quantity'], errors='coerce'),
                'ts_ms': pd.to_numeric(chunk['transact_time'], errors='coerce'),
                'is_buyer_maker': chunk['is_buyer_maker'].astype(str).str.lower().isin(['true', '1']),
            }).dropna()
            q = q[(q.price > 0) & (q.qty > 0)]
            if q.empty: continue
            monthly += len(q); total += len(q)

            prices = q['price'].to_numpy(dtype='float64')
            qtys = q['qty'].to_numpy(dtype='float64')
            ts_ns_arr = q['ts_ms'].to_numpy(dtype='int64') * 1_000_000
            is_sell = q['is_buyer_maker'].to_numpy(dtype=bool)
            usd_arr = qtys * contract_usd if market == 'futures/cm' else prices * qtys
            sign_arr = np.where(is_sell, -1.0, 1.0)
            signed_arr = sign_arr * usd_arr
            buy_arr = np.where(sign_arr > 0, usd_arr, 0.0)
            n_chunk = len(prices)

            cum_usd = cur_overshoot + np.cumsum(usd_arr, dtype='float64')
            k_max = int(cum_usd[-1] // threshold)
            if k_max == 0:
                if cur_trades == 0:
                    cur_open = float(prices[0]); cur_open_ts = int(ts_ns_arr[0])
                cur_high = max(cur_high, float(prices.max()))
                cur_low = min(cur_low, float(prices.min()))
                cur_usd += float(usd_arr.sum()); cur_signed += float(signed_arr.sum())
                cur_buys += float(buy_arr.sum()); cur_trades += n_chunk
                cur_overshoot = float(cum_usd[-1])
            else:
                levels = threshold * np.arange(1, k_max + 1, dtype='float64')
                close_ix = np.unique(np.searchsorted(cum_usd, levels, side='left'))
                close_ix = close_ix[close_ix < n_chunk]
                starts = np.r_[0, close_ix[:-1] + 1]
                end = int(close_ix[-1]) + 1
                b_open = prices[starts]; b_close = prices[close_ix]
                b_high = np.maximum.reduceat(prices[:end], starts); b_low = np.minimum.reduceat(prices[:end], starts)
                b_usd = np.add.reduceat(usd_arr[:end], starts); b_signed = np.add.reduceat(signed_arr[:end], starts)
                b_buys = np.add.reduceat(buy_arr[:end], starts)
                b_trades = (close_ix - starts + 1).astype(np.int64)
                b_open_ts = ts_ns_arr[starts]; b_close_ts = ts_ns_arr[close_ix]
                if cur_trades > 0:
                    b_open[0] = cur_open; b_high[0] = max(cur_high, float(b_high[0])); b_low[0] = min(cur_low, float(b_low[0]))
                    b_usd[0] += cur_usd; b_signed[0] += cur_signed; b_buys[0] += cur_buys
                    b_trades[0] += cur_trades; b_open_ts[0] = cur_open_ts
                for j in range(len(close_ix)):
                    ct = int(b_close_ts[j])
                    if ct <= last_ts_ns: ct = last_ts_ns + 1
                    last_ts_ns = ct
                    dur = max((ct - int(b_open_ts[j])) / 1e9, 1e-3)
                    bars.append((ct, float(b_open[j]), float(b_high[j]), float(b_low[j]), float(b_close[j]),
                                 float(b_usd[j]), float(b_signed[j]), float(b_buys[j]), int(b_trades[j]), dur))
                cur_overshoot = float(cum_usd[-1] - np.floor(cum_usd[close_ix[-1]] / threshold) * threshold)
                if end < n_chunk:
                    cur_open = float(prices[end]); cur_high = float(prices[end:].max()); cur_low = float(prices[end:].min())
                    cur_open_ts = int(ts_ns_arr[end]); cur_usd = float(usd_arr[end:].sum())
                    cur_signed = float(signed_arr[end:].sum()); cur_buys = float(buy_arr[end:].sum()); cur_trades = n_chunk - end
                else:
                    cur_open = np.nan; cur_high = -np.inf; cur_low = np.inf; cur_open_ts = -1
                    cur_usd = 0.0; cur_signed = 0.0; cur_buys = 0.0; cur_trades = 0
        print('Binance', asset, path.name, 'aggTrades:', monthly, 'dollarBars:', len(bars))

    if not bars:
        raise ValueError('No completed dollar bars')
    df = pd.DataFrame(bars, columns=[
        'timestamp_ns', 'open', 'high', 'low', 'close',
        'volume', 'signed_notional_usd', 'buy_notional_usd', 'trade_count', 'duration_sec'
    ])
    df['timestamp'] = pd.to_datetime(df['timestamp_ns'], unit='ns', utc=True)
    df['notional_usd'] = df['volume']
    df = df.drop(columns=['timestamp_ns'])
    out = Path(outfile); out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out, index=False)
    audit = {
        'asset': asset, 'market': market, 'source_files': len(files),
        'agg_trades': total, 'dollar_bars': len(df), 'dollar_bar_target_usd': threshold,
        'nonincreasing_aggregate_id_transitions': id_disorder_count,
        'aggregate_id_disorder_samples': id_disorders,
    }
    audit_path = out.with_suffix('.aggtrade_id_audit.json')
    audit_path.write_text(json.dumps(audit, indent=2))
    print('Dollar bars:', len(df), 'aggTrades:', total, 'saved:', out, 'ID anomalies:', id_disorder_count)
    return out

now = pd.Timestamp.now(tz='UTC')
last_complete = str(now.tz_localize(None).to_period('M') - 1) if CFG['binance_end'] is None else CFG['binance_end']
months = _month_range(CFG['binance_start'], last_complete)

def file_fingerprint(files, kind='binance', market=None, symbol=None):
    fingerprints = []
    for p in files:
        p = Path(p)
        if kind == 'binance' and CFG['verify_binance_sha256']:
            marker = p.with_suffix(p.suffix + '.sha256_verified.json')
            verified = json.loads(marker.read_text())
            if (verified['size'] != p.stat().st_size or verified['mtime_ns'] != p.stat().st_mtime_ns):
                raise RuntimeError(f'Stale source verification marker: {p}')
            digest = verified['expected_sha256']
        else:
            digest = _sha256_file(p)
        fingerprints.append((p.name, digest))
    contract = {
        'files': fingerprints, 'kind': kind, 'market': market, 'symbol': symbol,
        'dollar_bar_target_usd': CFG['dollar_bar_target_usd'],
        'binance_contract_usd': CFG['binance_contract_usd'],
        'pipeline_version': 'v8_dollar_btc_only', 'timestamp_semantics': CFG['timestamp_semantics']
    }
    return hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()

def build_cached_binance_asset(asset, market, symbol, archive, output):
    files = []
    for m in months:
        try:
            p = fetch_binance_month(m, sym=symbol, market=market, archive=archive)
        except FileNotFoundError:
            raise RuntimeError(f'Missing Binance month INSIDE active {asset} history: {m}')
        files.append(p)
    if not files:
        raise ValueError(f'No Binance monthly aggTrades archives for {asset}')
    fingerprint = file_fingerprint(files, kind='binance', market=market, symbol=symbol)
    path = Path(output); manifest = path.with_suffix('.source_manifest.json')
    try:
        reuse = path.exists() and json.loads(manifest.read_text())['hash'] == fingerprint
    except (OSError, ValueError, KeyError):
        reuse = False
    if not reuse:
        convert_binance_aggtrades_to_dollar_bars(files, path, market=market, asset=asset)
        manifest.write_text(json.dumps({
            'hash': fingerprint, 'market': market, 'symbol': symbol,
            'bar_type': 'dollar_bar', 'dollar_bar_target_usd': CFG['dollar_bar_target_usd']
        }, indent=2))
    else:
        print('Reusing cached', asset, path)
    return path

btc_path = build_cached_binance_asset(
    'BTCUSD', CFG['binance_market'], CFG['binance_symbol'],
    CFG['binance_archive_dir'], CFG['tick_bar_cache']
)
paths = [btc_path]
assert len(paths) == 1 and paths[0].exists(), 'BTC-only source required'

coverage = {}
d = load_asset_file(btc_path)
gaps = d.index.to_series().diff().dt.total_seconds()
coverage['BTCUSD'] = {
    'bars': len(d), 'start': str(d.index.min()), 'end': str(d.index.max()),
    'outage_gaps_gt_max': int((gaps.dropna() > CFG['max_gap_seconds']).sum()),
    'median_bar_seconds': float(gaps.dropna().median()) if len(gaps.dropna()) else 0.,
    'largest_gap_seconds': float(gaps.max()) if len(gaps.dropna()) else 0.,
    'source': str(btc_path), 'role': 'btc_only_dollar_bars'
}
start, end = d.index.min(), d.index.max()
del d, gaps; gc.collect()

sealed_start = end - pd.DateOffset(months=int(CFG['sealed_months']))
cal_start = sealed_start - pd.DateOffset(months=int(CFG['cal_months']))
train_end = cal_start - pd.DateOffset(months=int(CFG['val_months']))
train_start = train_end - pd.DateOffset(years=int(CFG['train_years']))
cuts = [train_end, cal_start, sealed_start]
if train_start < start:
    raise RuntimeError('Insufficient BTCUSD history for 5y train + 12m holdout')
CFG['train_start_utc'] = str(train_start); CFG['train_end_utc'] = str(train_end)
print('STRICT UTC folds:', train_start, cuts, end, '\nCoverage:', json.dumps(coverage, indent=2))

In [ ]:
# Cell 6 — Disk-backed Dollar-Bar features/targets, MTF closed candles, and lazy dataset
from torch.utils.data import Dataset, DataLoader
import gc

def _source_fingerprint(p, chunk=2**20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

_cache_inputs = [(str(p), _source_fingerprint(p)) for p in paths]
_cache_contract = {
    'sources': _cache_inputs, 'features': FEATURES,
    'code_version': 'v8_btc_dollar_bars_r2',
    'cfg': {k: CFG[k] for k in (
        'lookback', 'purge_bars', 'horizons', 'train_years', 'val_months',
        'cal_months', 'sealed_months', 'max_gap_seconds', 'timestamp_semantics',
        'mtf_timeframes', 'mtf_window', 'stress_vol_quantile', 'barrier_vol_multiplier',
        'neutral_move_bps', 'label_vol_multiplier', 'reference_cost_bps',
        'path_knots', 'validation_report_months', 'trailing_feature_window',
        'dollar_bar_target_usd'
    )}
}
CACHE_FINGERPRINT = hashlib.sha256(json.dumps(_cache_contract, sort_keys=True).encode()).hexdigest()[:20]
cache = Path(CFG['cache_dir']) / CACHE_FINGERPRINT
cache.mkdir(parents=True, exist_ok=True)
print('Verified cache namespace:', CACHE_FINGERPRINT)

def causal_feature_normalize(frame, center, scale, gap_seconds=None):
    normalized = (frame[FEATURES] - center) / scale
    w = int(CFG['trailing_feature_window'])
    if w > 1 and len(normalized) > 1:
        cols = [c for c in FEATURES if c not in ('spread_present', 'volume_present', 'tod_sin', 'tod_cos')]
        ts = frame.index.to_numpy(dtype='datetime64[ns]')
        steps = np.diff(ts).astype('timedelta64[ns]').astype(np.int64)
        gap_ns = int((gap_seconds if gap_seconds is not None else float(CFG['max_gap_seconds'])) * 1e9)
        groups = np.r_[0, np.cumsum(steps > gap_ns)]
        vals = normalized[cols]
        rolled = vals.groupby(groups, sort=False).rolling(w, min_periods=min(w, 48))
        mu = rolled.mean().reset_index(level=0, drop=True)
        sigma = rolled.std().reset_index(level=0, drop=True).clip(lower=.5)
        transformed = (vals - mu) / sigma
        normalized.loc[:, cols] = transformed.where(mu.notna(), vals)
    return normalized.clip(-12, 12).fillna(0.).to_numpy('float32')

TF_NAMES = list(CFG['mtf_timeframes'])
TF_FREQS = list(CFG['mtf_timeframes'].values())
TF_SECONDS = np.array([pd.Timedelta(x).total_seconds() for x in TF_FREQS], dtype=np.int64)

def build_closed_timeframe_features(d, freq):
    # Dollar bars carry CLOSE timestamps, so floor/groupby by [start, end) -> right-labeled close time.
    freq_delta = pd.Timedelta(freq)
    o = d[['open', 'high', 'low', 'close']].resample(freq, closed='right', label='right', origin='epoch').agg(
        {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'}
    )
    for field in ('volume', 'notional_usd', 'signed_notional_usd', 'buy_notional_usd', 'trade_count', 'duration_sec'):
        if field in d.columns:
            o[field] = d[field].resample(freq, closed='right', label='right', origin='epoch').sum(min_count=1)
    o = o.dropna(subset=['open', 'high', 'low', 'close']).copy()
    if not o.empty:
        # Ensure MTF bars only become visible at or after the underlying dollar bars' close times
        assert o.index.max() <= d.index.max() + freq_delta
    tf = causal_features(o, max_gap_seconds=max(float(CFG['max_gap_seconds']), freq_delta.total_seconds() * 2.0))
    return tf

def tf_training_samples(tf, cut):
    x = tf.loc[(tf.index >= train_start) & (tf.index < cut), FEATURES]
    return x.iloc[::max(1, len(x) // 25000)].copy() if len(x) else pd.DataFrame(columns=FEATURES)

H = CFG['horizons']; lb = CFG['lookback']; purge = CFG['purge_bars']
_cache_ready = cache / '.cache_ready.json'
_required_npy = (
    [cache / f'BTCUSD_{x}.npy' for x in ('features', 'targets', 'times', 'causal_stress', 'decision_mask', 'barrier_mask', 'mtf_stops')]
    + [cache / f'BTCUSD_fold{k}.npy' for k in range(4)]
    + [cache / f'BTCUSD_{tf}_{sfx}.npy' for tf in TF_NAMES for sfx in ('features', 'times', 'valid')]
)
if _cache_ready.exists() and all(p.exists() for p in _required_npy):
    _meta = json.loads(_cache_ready.read_text())
    med = pd.Series(_meta['med']); iqr = pd.Series(_meta['iqr'])
    mtf_scalers = {k: (pd.Series(v['med']), pd.Series(v['iqr'])) for k, v in _meta['mtf_scalers'].items()}
    counts = {int(k): int(v) for k, v in _meta['counts'].items()}
    assets = ['BTCUSD']
    print('Reusing verified Dollar-Bar tensor cache:', cache, counts)
else:
    d = load_asset_file(btc_path)
    d = d.loc[d.index >= train_start - pd.Timedelta(days=35)]
    f = causal_features(d)
    tr = f.loc[(f.index >= train_start) & (f.index < cuts[0]), FEATURES]
    if tr.empty:
        raise ValueError('No training rows for scaler')
    scaler_samples = tr.iloc[::max(1, len(tr) // 100000)].copy()
    med = scaler_samples.median().fillna(0.)
    iqr = (scaler_samples.quantile(.75) - scaler_samples.quantile(.25)).replace(0, 1.).fillna(1.)
    for _c in ('spread_present', 'volume_present', 'tod_sin', 'tod_cos'):
        if _c in med.index: med[_c] = 0.; iqr[_c] = 1.
    del scaler_samples, tr

    mtf_scalers = {}
    for tf_name, tf_freq in CFG['mtf_timeframes'].items():
        tf = build_closed_timeframe_features(d, tf_freq)
        sample = tf_training_samples(tf, cuts[0])
        if sample.empty:
            raise ValueError(f'No training samples for {tf_name}')
        tf_med = sample.median().fillna(0.)
        tf_iqr = (sample.quantile(.75) - sample.quantile(.25)).replace(0, 1.).fillna(1.)
        for _c in ('spread_present', 'volume_present', 'tod_sin', 'tod_cos'):
            if _c in tf_med.index: tf_med[_c] = 0.; tf_iqr[_c] = 1.
        mtf_scalers[tf_name] = (tf_med, tf_iqr)
        tf_z = causal_feature_normalize(tf, tf_med, tf_iqr, gap_seconds=max(float(CFG['max_gap_seconds']), pd.Timedelta(tf_freq).total_seconds() * 2.0))
        tf_valid = np.isfinite(tf[['ret1', 'vol12', 'vol48']].to_numpy()).all(1).astype('bool')
        np.save(cache / f'BTCUSD_{tf_name}_features.npy', tf_z, allow_pickle=False)
        np.save(cache / f'BTCUSD_{tf_name}_times.npy', tf.index.to_numpy('datetime64[ns]'), allow_pickle=False)
        np.save(cache / f'BTCUSD_{tf_name}_valid.npy', tf_valid, allow_pickle=False)
        del tf, sample, tf_z, tf_valid

    y = build_labels(d, H)
    z = causal_feature_normalize(f, med, iqr)
    spread_valid = f['spread_present'].fillna(0).clip(0, 1).to_numpy('float32')
    regime_extra = np.column_stack([
        (np.log1p(np.maximum(f['spread_rel'].fillna(0).to_numpy(), 0) * 1e4) / np.log1p(50)).clip(0, 1),
        (f['jump_proxy'].fillna(0).to_numpy() > 5).astype('float32'),
        f['entropy24'].fillna(0).to_numpy().clip(0, 1)
    ]).astype('float32')
    regime_extra_mask = np.column_stack([spread_valid, np.ones(len(d), dtype='float32'), np.ones(len(d), dtype='float32')]).astype('float32')

    targets = np.column_stack([
        y[[f'ret_{h}' for h in H]].to_numpy('float32'),
        y[[f'dir_{h}' for h in H]].to_numpy('float32'),
        y[[f'mfe_{h}' for h in H]].to_numpy('float32'),
        y[[f'mae_{h}' for h in H]].to_numpy('float32'),
        y[[f'rv_{h}' for h in H]].to_numpy('float32'),
        y[['decision', 'regime_trend', 'regime_high_vol', 'regime_transition']].to_numpy('float32'),
        y[[f'pattern_{h}' for h in H]].to_numpy('float32'),
        y[[f'jump_{h}' for h in H]].to_numpy('float32'),
        y[[f'path_{h}' for h in H]].to_numpy('float32'),
        regime_extra, regime_extra_mask,
        y[[f'{kind}_{h}' for h in H for kind in ('false_breakout', 'reversal', 'continuation')]].to_numpy('float32'),
        y[[f'path_increment_{k}' for k in CFG['path_knots']]].to_numpy('float32'),
        y[[f'barrier_{h}' for h in H]].to_numpy('float32')
    ])

    dates = f.index.to_numpy(dtype='datetime64[ns]'); ns = len(dates)
    folds = np.searchsorted(np.asarray(cuts, dtype='datetime64[ns]'), dates, side='right')
    decision_valid = np.isfinite(targets[:, 5 * len(H)])
    barrier_valid = np.isfinite(targets[:, -len(H):])
    base_ok = np.isfinite(targets).copy()
    base_ok[:, 5 * len(H)] = True
    base_ok[:, -len(H):] = True
    valid = base_ok.all(1)
    np.save(cache / 'BTCUSD_decision_mask.npy', decision_valid.astype('bool'), allow_pickle=False)
    np.save(cache / 'BTCUSD_barrier_mask.npy', barrier_valid.astype('bool'), allow_pickle=False)
    targets[:, 5 * len(H)] = np.where(decision_valid, targets[:, 5 * len(H)], 1.)
    targets[:, -len(H):] = np.where(barrier_valid, targets[:, -len(H):], 1.)
    valid[:lb + purge] = False; valid[-max(H) - purge:] = False
    core = np.isfinite(f[['ret1', 'vol12', 'vol48']].to_numpy()).all(1)
    run = np.convolve((~core).astype('int32'), np.ones(lb, dtype='int32'), mode='full')[:ns]
    valid &= (run == 0)

    # Outage-free continuity over [i - lb + 1, i + max(H)]
    steps_ns = np.diff(dates).astype('timedelta64[ns]').astype(np.int64)
    gap_ns = int(float(CFG['max_gap_seconds']) * 1e9)
    gap_prefix = np.r_[0, np.cumsum((steps_ns > gap_ns).astype(np.int64))]
    connected_span = np.zeros(ns, dtype=bool)
    hh = max(H)
    connected_span[lb - 1:ns - hh] = (gap_prefix[lb - 1 + hh:] - gap_prefix[:ns - hh - (lb - 1)]) == 0
    valid &= connected_span

    indexes = np.where(valid)[0]
    indexes = indexes[(indexes + purge < ns) & (indexes - purge >= 0)]
    indexes = indexes[dates[indexes] >= train_start.tz_localize(None).to_datetime64()]
    indexes = indexes[
        (folds[indexes] == folds[indexes - purge]) &
        (folds[indexes] == folds[indexes + purge]) &
        (folds[indexes] == folds[indexes - lb + 1]) &
        (folds[indexes] == folds[indexes + max(H)])
    ]

    train_vol = f.loc[(f.index >= train_start) & (f.index < train_end), 'vol48'].to_numpy(dtype=np.float64)
    train_vol = train_vol[np.isfinite(train_vol)]
    stress_threshold = float(np.quantile(train_vol, CFG['stress_vol_quantile'])) if len(train_vol) > 100 else float('inf')
    stress = (f['vol48'].fillna(0).to_numpy(dtype=np.float64) >= stress_threshold).astype('uint8')
    np.save(cache / 'BTCUSD_causal_stress.npy', stress, allow_pickle=False)
    np.save(cache / 'BTCUSD_features.npy', z, allow_pickle=False)
    np.save(cache / 'BTCUSD_targets.npy', targets.astype('float32'), allow_pickle=False)
    np.save(cache / 'BTCUSD_times.npy', dates, allow_pickle=False)

    assets = ['BTCUSD']
    counts = {}
    for k in range(4):
        chosen = indexes[folds[indexes] == k].astype('int32')
        np.save(cache / f'BTCUSD_fold{k}.npy', chosen, allow_pickle=False)
        counts[k] = len(chosen)
    if counts[0] < CFG['min_train_sequences'] or any(counts[k] == 0 for k in (1, 2, 3)):
        raise ValueError(f'Insufficient valid samples per split: {counts}')
    mtf_stops = np.column_stack([
        np.searchsorted(np.load(cache / f'BTCUSD_{tf}_times.npy', mmap_mode='r'), dates, side='right').astype('int32')
        for tf in TF_NAMES
    ])
    np.save(cache / 'BTCUSD_mtf_stops.npy', mtf_stops, allow_pickle=False)
    _cache_ready.write_text(json.dumps({
        'med': med.to_dict(), 'iqr': iqr.to_dict(),
        'mtf_scalers': {k: {'med': v[0].to_dict(), 'iqr': v[1].to_dict()} for k, v in mtf_scalers.items()},
        'counts': counts
    }))
    print('BTCUSD Dollar-Bar samples per fold:', counts)
    del d, f, y, z, targets, dates, folds, indexes, mtf_stops; gc.collect()

class LazyFold(Dataset):
    def __init__(self, k, validation_role='selection'):
        self.X = np.load(cache / 'BTCUSD_features.npy', mmap_mode='r')
        self.Y = np.load(cache / 'BTCUSD_targets.npy', mmap_mode='r')
        self.T = np.load(cache / 'BTCUSD_times.npy', mmap_mode='r')
        self.DM = np.load(cache / 'BTCUSD_decision_mask.npy', mmap_mode='r')
        self.BM = np.load(cache / 'BTCUSD_barrier_mask.npy', mmap_mode='r')
        self.MS = np.load(cache / 'BTCUSD_mtf_stops.npy', mmap_mode='r')
        self.MX = [np.load(cache / f'BTCUSD_{tf}_features.npy', mmap_mode='r') for tf in TF_NAMES]
        self.MV = [np.load(cache / f'BTCUSD_{tf}_valid.npy', mmap_mode='r') for tf in TF_NAMES]
        raw_ids = np.load(cache / f'BTCUSD_fold{k}.npy', mmap_mode='r')
        if k == 1:
            t_fold = np.asarray(self.T[raw_ids], dtype='datetime64[ns]')
            report_cut = (cal_start - pd.DateOffset(months=int(CFG['validation_report_months']))).tz_localize(None).to_datetime64()
            if validation_role == 'selection':
                keep = t_fold < report_cut
                ids = np.asarray(raw_ids[keep], dtype='int32')
                if len(ids) > purge:
                    ids = ids[:-purge]
                max_val = int(CFG['stage_max_val_steps']) * int(CFG['batch_size']) if CFG['stage_max_val_steps'] else len(ids)
                if len(ids) > max_val:
                    ids = ids[np.linspace(0, len(ids) - 1, max_val, dtype='int64')]
            elif validation_role == 'report':
                ids = np.asarray(raw_ids[t_fold >= report_cut], dtype='int32')
            else:
                raise ValueError(validation_role)
            self.ids = ids
        else:
            self.ids = raw_ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        i = int(self.ids[idx])
        x = np.array(self.X[i - lb + 1:i + 1], copy=True)
        y = np.array(self.Y[i], copy=True)
        regime_full = np.concatenate([y[5 * len(H) + 1:5 * len(H) + 4], y[8 * len(H) + 4:8 * len(H) + 7], np.zeros(1, dtype='float32')])
        regime_full_mask = np.concatenate([np.ones(3, dtype='float32'), y[8 * len(H) + 7:8 * len(H) + 10], np.zeros(1, dtype='float32')])
        pattern_detail = y[8 * len(H) + 10:11 * len(H) + 10].reshape(len(H), 3)
        p_start = 11 * len(H) + 10; p_end = p_start + len(CFG['path_knots'])
        future_path = y[p_start:p_end]
        barrier_targets = y[p_end:p_end + len(H)]

        stops = self.MS[i]
        w = int(CFG['mtf_window']); nfeat = len(FEATURES)
        mtf = np.zeros((len(TF_NAMES), w, nfeat), dtype='float32')
        mtf_mask = np.zeros((len(TF_NAMES), w), dtype='bool')
        for j in range(len(TF_NAMES)):
            stop = int(stops[j])
            if stop > 0:
                start_j = max(0, stop - w); length = stop - start_j
                mtf[j, -length:] = np.array(self.MX[j][start_j:stop], copy=True)
                mtf_mask[j, -length:] = np.array(self.MV[j][start_j:stop], copy=True)

        return (
            torch.from_numpy(x),
            torch.from_numpy(y[:len(H)]),
            torch.from_numpy(y[len(H):2 * len(H)].astype('int64')),
            torch.from_numpy(y[2 * len(H):3 * len(H)]),
            torch.from_numpy(y[3 * len(H):4 * len(H)]),
            torch.from_numpy(y[4 * len(H):5 * len(H)]),
            torch.tensor(int(y[5 * len(H)]), dtype=torch.long),
            torch.from_numpy(regime_full),
            torch.from_numpy(y[5 * len(H) + 4:6 * len(H) + 4]),
            torch.from_numpy(y[6 * len(H) + 4:7 * len(H) + 4]),
            torch.from_numpy(y[7 * len(H) + 4:8 * len(H) + 4]),
            torch.from_numpy(regime_full_mask),
            torch.from_numpy(pattern_detail),
            torch.from_numpy(future_path),
            torch.from_numpy(barrier_targets.astype('int64')),
            torch.from_numpy(mtf),
            torch.from_numpy(mtf_mask),
            torch.tensor(bool(self.DM[i])),
            torch.from_numpy(np.asarray(self.BM[i], dtype=bool).copy()),
        )

    def ref(self, idx):
        i = int(self.ids[idx])
        return 'BTCUSD', str(pd.Timestamp(self.T[i], tz='UTC'))

def training_sample_weights():
    ids = np.load(cache / 'BTCUSD_fold0.npy', mmap_mode='r')
    timestamps = np.load(cache / 'BTCUSD_times.npy', mmap_mode='r')
    stress = np.load(cache / 'BTCUSD_causal_stress.npy', mmap_mode='r')
    t = np.asarray(timestamps[ids], dtype='datetime64[ns]')
    age_days = np.maximum((train_end.tz_localize(None).to_datetime64() - t) / np.timedelta64(1, 'D'), 0.)
    recent = np.exp2(-age_days / float(CFG['recent_half_life_days']))
    weight = float(CFG['recent_uniform_floor']) + (1. - float(CFG['recent_uniform_floor'])) * recent
    flags = np.asarray(stress[ids], dtype=np.float64)
    weight *= np.where(flags > 0., min(float(CFG['stress_boost']), float(CFG['stress_cap'])), 1.)
    weight = weight / max(weight.sum(), 1e-12)
    print('TRAIN-ONLY recency/stress sampling:', json.dumps({
        'asset': 'BTCUSD', 'train_rows': int(len(ids)), 'stress_rows': int(flags.sum()),
        'weight_min': float(weight.min()), 'weight_max': float(weight.max()),
        'recent_365d_rows': int((age_days <= 365.).sum())
    }))
    return torch.from_numpy(weight)

class DeterministicEpochSampler(torch.utils.data.Sampler):
    def __init__(self, weights, num_samples):
        w = weights.double().numpy() if torch.is_tensor(weights) else np.asarray(weights, dtype=np.float64)
        self.cdf = np.cumsum(w, dtype=np.float64)
        self.cdf[-1] = 1.0
        self.total = int(num_samples)
        self.num_draw = self.total
        self.epoch_seed = 47; self.start = 0
    def set_epoch(self, phase, stage, epoch, completed_steps=0):
        token = f'{SEED}:{phase}:{stage}:{epoch}'
        self.epoch_seed = int(hashlib.sha256(token.encode()).hexdigest()[:15], 16)
        cap = int(CFG['max_train_steps_per_epoch']) * int(CFG['batch_size']) if CFG['max_train_steps_per_epoch'] else self.total
        self.num_draw = min(self.total, cap)
        self.start = int(completed_steps) * int(CFG['batch_size'])
        if not 0 <= self.start <= self.num_draw:
            raise RuntimeError('Invalid resume batch offset')
    def __iter__(self):
        rng = np.random.default_rng(self.epoch_seed)
        u = rng.random(self.num_draw)
        seq = np.clip(np.searchsorted(self.cdf, u, side='right'), 0, len(self.cdf) - 1)
        return iter(seq[self.start:].tolist())
    def __len__(self):
        return max(0, self.num_draw - self.start)

_FOLD_LOADERS = {}
def dl(k, shuffle=False, validation_role='selection'):
    key = (int(k), bool(shuffle), validation_role)
    if key not in _FOLD_LOADERS:
        workers = int(CFG['loader_workers'])
        ds = LazyFold(k, validation_role)
        sampler = (DeterministicEpochSampler(training_sample_weights(), num_samples=len(ds))
                   if k == 0 and shuffle else None)
        kwargs = {
            'batch_size': CFG['batch_size'],
            'shuffle': bool(shuffle and sampler is None),
            'sampler': sampler,
            'num_workers': workers,
            'pin_memory': DEVICE.type == 'cuda',
            'generator': torch.Generator().manual_seed(SEED + int(k))
        }
        if workers > 0:
            kwargs.update(persistent_workers=bool(CFG['persistent_workers']),
                          prefetch_factor=int(CFG['prefetch_factor']))
        _FOLD_LOADERS[key] = DataLoader(ds, **kwargs)
    return _FOLD_LOADERS[key]

def move_batch(pack):
    return tuple(z.to(DEVICE, non_blocking=DEVICE.type == 'cuda') for z in pack)

t_arr = np.load(cache / 'BTCUSD_targets.npy', mmap_mode='r')
ids0 = np.load(cache / 'BTCUSD_fold0.npy', mmap_mode='r')
subset = np.asarray(ids0[::max(1, len(ids0) // 10000)], dtype=int)
trvals = np.array(t_arr[subset], copy=True)
ret_scale = max(float(np.median(np.abs(trvals[:, len(H) - 1]))), 1e-5)
rv_scale = max(float(np.median(trvals[:, 5 * len(H) - 1])), 1e-8)

def _inv_sqrt_weights(labels_1d):
    counts = np.bincount(labels_1d.astype('int64'), minlength=3).astype('float64')
    counts = np.maximum(counts, 1.0)
    w = (counts.sum() / (3.0 * counts)) ** 0.5
    w = np.clip(w / w.mean(), 0.5, 2.5)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE), counts.astype('int64').tolist()

dm_arr = np.load(cache / 'BTCUSD_decision_mask.npy', mmap_mode='r')
bm_arr = np.load(cache / 'BTCUSD_barrier_mask.npy', mmap_mode='r')
dm0 = np.asarray(dm_arr[subset], dtype=bool)
bm0 = np.asarray(bm_arr[subset], dtype=bool)
dec_w, dec_cnt = _inv_sqrt_weights(trvals[dm0, 5 * len(H)] if dm0.any() else trvals[:, 5 * len(H)])
dir_w, dir_cnt = _inv_sqrt_weights(trvals[:, len(H):2 * len(H)].reshape(-1))
p_end = 11 * len(H) + 10 + len(CFG['path_knots'])
bar_vals = trvals[:, p_end:p_end + len(H)]
bar_w, bar_cnt = _inv_sqrt_weights(bar_vals[bm0] if bm0.any() else bar_vals.reshape(-1))
del trvals, dm0, bm0, bar_vals
print('Train-only scale:', ret_scale, rv_scale)
print('Train-only class weights (SHORT/WAIT/LONG):', {
    'decision': (dec_cnt, [round(float(x), 3) for x in dec_w.cpu()]),
    'direction': (dir_cnt, [round(float(x), 3) for x in dir_w.cpu()]),
    'barrier': (bar_cnt, [round(float(x), 3) for x in bar_w.cpu()]),
})

# Exhaustive MTF as-of audit
_base = np.load(cache / 'BTCUSD_times.npy', mmap_mode='r')
for _k in range(4):
    _ids = np.load(cache / f'BTCUSD_fold{_k}.npy', mmap_mode='r')
    if not len(_ids): continue
    _asofs = np.asarray(_base[_ids])
    for _tf in TF_NAMES:
        _tf_times = np.load(cache / f'BTCUSD_{_tf}_times.npy', mmap_mode='r')
        _stops = np.searchsorted(_tf_times, _asofs, side='right')
        _has = _stops > 0
        if _has.any():
            assert np.all(_tf_times[_stops[_has] - 1] <= _asofs[_has]), ('BTCUSD', _k, _tf)
print('Exhaustive MTF as-of index audit PASS')
assert len(dl(1, False, validation_role='report').dataset) > 0, 'Empty independent validation report fold'

## 3. Original Quant Transformer + SSM + MTF + Sparse MoE (BTC-Only)
Retains the full `UniversalQuantModel` backbone (Conv + Causal Transformer + GRU + Diagonal SSM), `MultiTimeframeFusion`, 8-expert Top-2 Sparse MoE, and all 20 probabilistic heads (`direction`, `quantiles`, `rv`, `excursions`, `barrier`, `pattern`, `pattern_detail`, `jump`, `path`, `LowRankMultivariateNormal` path increments, `action_quantiles`, `decision`, `regime`, `ssl_prediction`). Unused cross-asset `DynamicTemporalGraph` has been removed.


In [ ]:
# Cell 10 — Original UniversalQuantModel (BTC-only, without unused Cross-Asset Graph)
class CausalDiagonalStateSpace(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.decay_logits = nn.Parameter(torch.linspace(.5, 5., dim))
        self.in_projection = nn.Linear(dim, dim)
        self.out_gate = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, h):
        B, T, D = h.shape
        alpha = torch.sigmoid(self.decay_logits).clamp(.05, .999)
        lags = torch.arange(T, device=h.device, dtype=h.dtype)
        impulse = (1 - alpha[:, None]) * alpha[:, None].pow(lags[None, :])
        states = F.conv1d(self.in_projection(h).transpose(1, 2),
                          impulse.flip(-1).unsqueeze(1), padding=T - 1, groups=D)[..., :T]
        return self.norm(states.transpose(1, 2) * torch.sigmoid(self.out_gate(h)))

class MultiTimeframeFusion(nn.Module):
    def __init__(self, features, hidden, heads, layers):
        super().__init__()
        drop = float(CFG.get('dropout', 0.25))
        self.encoders = nn.ModuleList([nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden, heads, hidden * 2, dropout=drop, batch_first=True, norm_first=True),
            layers, norm=nn.LayerNorm(hidden), enable_nested_tensor=False) for _ in TF_NAMES])
        self.pos_emb = nn.Parameter(torch.randn(1, CFG['mtf_window'], hidden) * .02)
        self.input = nn.ModuleList([nn.Sequential(nn.Linear(features, hidden), nn.LayerNorm(hidden)) for _ in TF_NAMES])
        self.tokens = nn.Parameter(torch.randn(len(TF_NAMES), hidden) * .02)
        self.query = nn.Linear(hidden, hidden)
        self.key = nn.Linear(hidden, hidden)
        self.value = nn.Linear(hidden, hidden)
        self.output = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.GELU(), nn.Dropout(drop), nn.LayerNorm(hidden))
        self.norm = nn.LayerNorm(hidden)
        self.scale = hidden ** -.5
        self.register_buffer('causal_mask', torch.triu(torch.ones(CFG['mtf_window'], CFG['mtf_window'], dtype=torch.bool), diagonal=1), persistent=False)
    def forward(self, query, mtf, mtf_mask):
        B, K, W, Fdim = mtf.shape
        assert K == len(TF_NAMES) and mtf_mask.shape == (B, K, W)
        tokens = []; present = []
        causal = self.causal_mask[:W, :W]
        for j in range(K):
            good = mtf_mask[:, j, :].bool()
            h = (self.input[j](mtf[:, j]) + self.tokens[j] + self.pos_emb[:, -W:]) * good.unsqueeze(-1).to(mtf.dtype)
            h = self.encoders[j](h, mask=causal)
            tokens.append(h[:, -1]); present.append(good.any(1))
        t = torch.stack(tokens, dim=1)
        available = torch.stack(present, dim=1)
        scores = (self.query(query)[:, None, :] * self.key(t)).sum(-1) * self.scale
        scores = scores.masked_fill(~available, -1e9)
        alpha = F.softmax(scores, dim=-1) * available.float()
        alpha = alpha / alpha.sum(-1, keepdim=True).clamp(min=1e-8)
        fused = (alpha.unsqueeze(-1) * self.value(t)).sum(1)
        blended = self.output(torch.cat([query, fused], dim=-1))
        return torch.where(available.any(1, keepdim=True), blended, query), alpha

class UniversalQuantModel(nn.Module):
    def __init__(self, features, hidden=192, nexperts=8, topk=2, nh=4,
                 transformer_layers=5, attention_heads=6, gru_layers=2, expert_width=512):
        super().__init__()
        drop = float(CFG.get('dropout', 0.25))
        self.nh = nh; self.topk = topk; self.npath = len(CFG['path_knots']); self.rank = CFG['path_rank']
        self.input = nn.Sequential(nn.Linear(features, hidden), nn.LayerNorm(hidden))
        self.pos_emb = nn.Parameter(torch.randn(1, CFG['lookback'], hidden) * .02)
        self.in_drop = nn.Dropout(min(0.15, drop * 0.5))
        self.conv = nn.Conv1d(hidden, hidden, 5, padding=4, groups=hidden)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(hidden, attention_heads, hidden * 4, dropout=drop, batch_first=True, norm_first=True),
            transformer_layers, norm=nn.LayerNorm(hidden), enable_nested_tensor=False
        )
        self.longrange = nn.GRU(hidden, hidden, num_layers=gru_layers, batch_first=True, dropout=drop if gru_layers > 1 else 0.)
        self.ssm = CausalDiagonalStateSpace(hidden)
        self.backbone_fusion = nn.Sequential(nn.Linear(hidden * 3, hidden), nn.GELU(), nn.Dropout(drop), nn.LayerNorm(hidden))
        self.mtf = MultiTimeframeFusion(features, hidden, attention_heads, CFG['mtf_encoder_layers'])
        self.ssl_predict = nn.Linear(hidden, features)
        self.regime = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, 7))
        self.router = nn.Sequential(nn.Linear(hidden + 7, hidden), nn.GELU(), nn.Linear(hidden, nexperts))
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, expert_width), nn.GELU(), nn.Dropout(drop),
                nn.Linear(expert_width, expert_width), nn.GELU(), nn.Dropout(drop),
                nn.Linear(expert_width, hidden)
            ) for _ in range(nexperts)
        ])
        self.moe_scale = nn.Parameter(torch.full((hidden,), 0.25))
        self.forecast_fusion = nn.Sequential(nn.Linear(hidden * 2 + 7, hidden), nn.GELU(), nn.Dropout(drop), nn.LayerNorm(hidden))
        self.ctx_norm = nn.LayerNorm(hidden)
        self.direction = nn.Linear(hidden, nh * 3)
        self.quantiles = nn.Linear(hidden, nh * 3)
        self.rv = nn.Linear(hidden, nh)
        self.excursions = nn.Linear(hidden, nh * 2)
        self.barrier = nn.Linear(hidden, nh * 3)
        self.pattern = nn.Linear(hidden, nh)
        self.pattern_detail = nn.Linear(hidden, nh * 3)
        self.jump = nn.Linear(hidden, nh)
        self.path = nn.Linear(hidden, nh)
        self.path_distribution = nn.Linear(hidden, self.npath * (2 + self.rank))
        _dq_in = hidden + nh * 9
        self.decision_quantiles = nn.Sequential(
            nn.LayerNorm(_dq_in), nn.Linear(_dq_in, hidden // 2), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden // 2, 9)
        )
        _dec_in = hidden + nh * 9 + 9
        self.decision = nn.Sequential(
            nn.LayerNorm(_dec_in), nn.Linear(_dec_in, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden // 2, 3)
        )
        self.register_buffer('causal_mask', torch.triu(torch.ones(CFG['lookback'], CFG['lookback'], dtype=torch.bool), diagonal=1), persistent=False)

    def forward(self, x, use_experts=True, ablated_expert=None, mtf=None, mtf_mask=None, use_mtf=True):
        B, T, _ = x.shape
        h = self.in_drop(self.input(x) + self.pos_emb[:, -T:])
        h = h + self.conv(h.transpose(1, 2))[..., :T].transpose(1, 2)
        causal = self.causal_mask[:T, :T]
        temporal = self.encoder(h, mask=causal)[:, -1]
        recurrent = self.longrange(h)[0][:, -1]
        state_space = self.ssm(h)[:, -1]
        z0 = self.backbone_fusion(torch.cat([temporal, recurrent, state_space], -1))
        if use_mtf and mtf is not None:
            assert mtf_mask is not None
            z0, mtf_attention = self.mtf(z0, mtf, mtf_mask)
        else:
            mtf_attention = torch.zeros(B, len(TF_NAMES), device=x.device, dtype=x.dtype)
        regime = self.regime(z0)
        route = self.router(torch.cat([z0, torch.sigmoid(regime)], -1))
        if self.training:
            jitter = 0.05 * route.detach().std(dim=-1, keepdim=True).clamp(min=1e-4) * torch.randn_like(route)
            top = (route + jitter).topk(min(self.topk, route.shape[-1]), dim=-1)
        else:
            top = route.topk(min(self.topk, route.shape[-1]), dim=-1)
        top_vals = route.gather(1, top.indices)
        gate = torch.zeros_like(route).scatter(1, top.indices, F.softmax(top_vals.float(), dim=-1).to(route.dtype))
        if ablated_expert is not None:
            gate = gate.clone(); gate[:, int(ablated_expert)] = 0.
            gate = gate / gate.sum(1, keepdim=True).clamp(min=1e-8)
        mixture = torch.zeros_like(z0)
        if use_experts:
            for j, expert in enumerate(self.experts):
                selected = torch.nonzero(gate[:, j] > 0, as_tuple=True)[0]
                if selected.numel():
                    expert_contribution = expert(z0.index_select(0, selected)) * gate[selected, j, None]
                    mixture = mixture.index_add(0, selected, expert_contribution.to(dtype=mixture.dtype))
        z = z0 + self.moe_scale * mixture
        ctx = self.ctx_norm(z + self.forecast_fusion(torch.cat([z, z0, torch.sigmoid(regime)], dim=-1)))
        direction = self.direction(ctx).reshape(B, self.nh, 3)
        raw = self.quantiles(ctx).reshape(B, self.nh, 3)
        median = raw[..., 1] * ret_scale
        quantiles = torch.stack([median - F.softplus(raw[..., 0]) * ret_scale, median, median + F.softplus(raw[..., 2]) * ret_scale], dim=-1)
        raw_path = self.path_distribution(ctx).reshape(B, self.npath, 2 + self.rank)
        path_mu = raw_path[..., 0] * ret_scale
        path_diag = (F.softplus(raw_path[..., 1]).clamp(max=10.) + .05) * ret_scale
        path_factors = torch.tanh(raw_path[..., 2:]) * ret_scale
        rv_pred = F.softplus(self.rv(ctx)) * rv_scale
        exc_pred = self.excursions(ctx).reshape(B, self.nh, 2) * ret_scale
        barrier_logits = self.barrier(ctx).reshape(B, self.nh, 3)
        direction_probs = F.softmax(direction, dim=-1).reshape(B, -1)
        barrier_probs = F.softmax(barrier_logits, dim=-1).reshape(B, -1)
        risk_feats = torch.cat([
            direction_probs, barrier_probs,
            torch.log1p(rv_pred / rv_scale),
            (exc_pred / ret_scale).reshape(B, -1)
        ], dim=-1)
        dq = self.decision_quantiles(torch.cat([ctx, risk_feats], dim=-1)).reshape(B, 3, 3)
        utility_median = dq[..., 1]
        action_quantiles = torch.stack([utility_median - F.softplus(dq[..., 0]), utility_median, utility_median + F.softplus(dq[..., 2])], dim=-1) * ret_scale
        decision = self.decision(torch.cat([ctx, risk_feats, action_quantiles.reshape(B, -1) / ret_scale], -1))
        return {
            'direction': direction, 'quantiles': quantiles, 'rv': rv_pred,
            'excursions': exc_pred,
            'barrier': barrier_logits, 'pattern': self.pattern(ctx),
            'pattern_detail': self.pattern_detail(ctx).reshape(-1, self.nh, 3),
            'jump': self.jump(ctx), 'path': self.path(ctx),
            'path_mu': path_mu, 'path_diag': path_diag, 'path_factors': path_factors,
            'action_quantiles': action_quantiles, 'decision': decision, 'regime': regime,
            'ssl_prediction': self.ssl_predict(z0), 'gate': gate,
            'router_probs': F.softmax(route, dim=-1), 'dispatch': (gate > 0).float(),
            'mtf_attention': mtf_attention
        }

def path_distribution_nll(o, target):
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        scale = float(ret_scale)
        mu = o['path_mu'].float() / scale
        factors = o['path_factors'].float() / scale
        diagonal = (o['path_diag'].float() / scale).square().clamp(min=1e-5)
        dist = torch.distributions.LowRankMultivariateNormal(loc=mu, cov_factor=factors, cov_diag=diagonal)
        nll = -dist.log_prob(target.float() / scale).mean()
    return nll

@torch.no_grad()
def path_forecast_quantiles(o):
    mu = o['path_mu'].cumsum(-1)
    diag_var = o['path_diag'].square().cumsum(-1)
    factor_sum = o['path_factors'].cumsum(1)
    var = diag_var + factor_sum.square().sum(-1)
    sd = var.clamp(min=1e-15).sqrt()
    return torch.stack([mu - 1.2815515655 * sd, mu, mu + 1.2815515655 * sd], dim=-1)

@torch.no_grad()
def sample_correlated_price_paths(o, starting_price, scenarios=64):
    with torch.autocast(device_type=o['path_mu'].device.type, enabled=False):
        dist = torch.distributions.LowRankMultivariateNormal(
            loc=o['path_mu'].float(), cov_factor=o['path_factors'].float(),
            cov_diag=o['path_diag'].float().square().clamp(min=1e-14)
        )
        increments = dist.sample((int(scenarios),))
        return float(starting_price) * increments.cumsum(-1).exp()

model = UniversalQuantModel(
    len(FEATURES), CFG['hidden'], CFG['experts'], CFG['top_k'], len(H),
    CFG['transformer_layers'], CFG['attention_heads'], CFG['gru_layers'], CFG['expert_width']
).to(DEVICE)
PARAMETERS = sum(p.numel() for p in model.parameters())
if DEVICE.type == 'cuda' and hasattr(torch, 'compile'):
    try:
        model.encoder.forward = torch.compile(model.encoder.forward)
        model.mtf.forward = torch.compile(model.mtf.forward)
        model.ssm.forward = torch.compile(model.ssm.forward)
        print('torch.compile enabled on static-shape encoder & mtf!')
    except Exception as _exc:
        print('torch.compile skipped:', _exc)
print('BTC-only Dollar-Bar trainable parameters:', PARAMETERS)

In [ ]:
# Cell 11 — 7-Stage Progressive Optimizer (SSL -> Regime -> Forecast -> Path -> Experts -> Decision -> Joint PCGrad)
def pinball(pred, y):
    tau = torch.tensor([.1, .5, .9], dtype=pred.dtype, device=pred.device)
    error = y.unsqueeze(-1) - pred
    return torch.maximum(tau * error, (tau - 1) * error).mean()

def cost_aware_utilities(r, m, n):
    terminal = r[:, -1]
    long = terminal - CFG['reference_cost_bps'] * 1e-4 - CFG['research_adverse_penalty'] * (-n[:, -1]).clamp(min=0)
    short = -terminal - CFG['reference_cost_bps'] * 1e-4 - CFG['research_adverse_penalty'] * m[:, -1].clamp(min=0)
    return torch.stack([short, torch.zeros_like(terminal), long], dim=-1)

def regime_loss(o, b):
    target = b[7]; mask = b[11]
    assert target.shape[1] == 7 and mask.shape == target.shape
    err = F.binary_cross_entropy_with_logits(o['regime'], target, reduction='none')
    return (err * mask).sum() / mask.sum().clamp(min=1)

def masked_mean(loss, mask):
    mask = mask.to(dtype=loss.dtype)
    return (loss * mask).sum() / mask.sum().clamp(min=1)

_HW = torch.tensor([1.35, 1.15, 0.90, 0.60][:len(H)], device=DEVICE, dtype=torch.float32)
_HW = _HW / _HW.mean()

def masked_barrier_loss(o, b):
    B = len(b[0]); nh = len(H)
    logits = o['barrier'].float().reshape(-1, 3)
    tgt = b[14].long().reshape(-1)
    ce = F.cross_entropy(logits, tgt, weight=bar_w, reduction='none',
                         label_smoothing=float(CFG.get('label_smoothing', 0.05)))
    probs = F.softmax(logits, dim=-1)
    opp = probs[:, 0] * (tgt == 2).float() + probs[:, 2] * (tgt == 0).float()
    losses = ((ce + 0.35 * opp).reshape(B, nh) * _HW.unsqueeze(0))
    return masked_mean(losses, b[18])

def masked_decision_loss(o, b):
    logits = o['decision'].float()
    tgt = b[6].long()
    ce = F.cross_entropy(logits, tgt, weight=dec_w, reduction='none',
                         label_smoothing=float(CFG.get('label_smoothing', 0.05)))
    probs = F.softmax(logits, dim=-1)
    opp_err = probs[:, 0] * (tgt == 2).float() + probs[:, 2] * (tgt == 0).float()
    return masked_mean(ce + 0.55 * opp_err, b[17])

def action_loss(o, b):
    util = cost_aware_utilities(b[1], b[3], b[4]).detach()
    classification = masked_decision_loss(o, b)
    quantile = pinball(o['action_quantiles'] / ret_scale, util / ret_scale)
    q_med = (o['action_quantiles'][..., 1].detach().float() / ret_scale).clamp(-5., 5.)
    q_prob = F.softmax(q_med * 2.0, dim=-1)
    d_logp = F.log_softmax(o['decision'].float(), dim=-1)
    kl_row = (q_prob * (q_prob.clamp(min=1e-8).log() - d_logp)).sum(-1)
    consistency = masked_mean(kl_row, b[17])
    return CFG['decision_weight'] * (classification + CFG['action_q_weight'] * quantile + 0.15 * consistency)

def router_balance(o):
    usage = o['dispatch'].mean(0).detach()
    probs = o['router_probs'].mean(0)
    n_exp = len(usage)
    load_bal = n_exp * (usage * probs).sum()
    uniform_kl = (probs * torch.log((probs * n_exp).clamp(min=1e-8))).sum()
    return load_bal + 0.25 * uniform_kl

def _weighted_direction_loss(o, d, ls):
    B = len(d); nh = len(H)
    logits = o['direction'].float().reshape(-1, 3)
    tgt = d.long().reshape(-1)
    ce = F.cross_entropy(logits, tgt, weight=dir_w, reduction='none', label_smoothing=ls)
    probs = F.softmax(logits, dim=-1)
    opp = probs[:, 0] * (tgt == 2).float() + probs[:, 2] * (tgt == 0).float()
    return ((ce + 0.35 * opp).reshape(B, nh) * _HW.unsqueeze(0)).mean()

def forecast_losses(o, b, include_regime=True, include_paths=True, include_route=True):
    r, d, m, n, v, decision, pattern, jump, path = b[1], b[2], b[3], b[4], b[5], b[6], b[8], b[9], b[10]
    ls = float(CFG.get('label_smoothing', 0.05))
    total = (
        CFG['direction_weight'] * _weighted_direction_loss(o, d, ls) +
        CFG['quantile_weight'] * pinball(o['quantiles'] / ret_scale, r / ret_scale) +
        CFG['rv_weight'] * F.smooth_l1_loss(torch.log1p(o['rv'] / rv_scale), torch.log1p(v / rv_scale)) +
        CFG['excursion_weight'] * .5 * (
            F.smooth_l1_loss(o['excursions'][..., 0] / ret_scale, m / ret_scale) +
            F.smooth_l1_loss(o['excursions'][..., 1] / ret_scale, n / ret_scale)
        ) +
        CFG['barrier_weight'] * masked_barrier_loss(o, b) +
        CFG['pattern_weight'] * F.binary_cross_entropy_with_logits(o['pattern'], pattern) +
        CFG['pattern_detail_weight'] * F.binary_cross_entropy_with_logits(o['pattern_detail'], b[12]) +
        CFG['jump_weight'] * F.binary_cross_entropy_with_logits(o['jump'], jump) +
        CFG['path_weight'] * F.smooth_l1_loss(o['path'], path)
    )
    if include_paths:
        total = total + CFG['path_nll_weight'] * path_distribution_nll(o, b[13])
    if include_regime:
        total = total + CFG['regime_weight'] * regime_loss(o, b)
    if include_route:
        total = total + CFG['route_balance_weight'] * router_balance(o)
    return total

def objective(o, b):
    return forecast_losses(o, b) + action_loss(o, b)

def joint_pcgrad_backward(predictive, decision_obj):
    params = [p for p in model.parameters() if p.requires_grad]
    a = torch.autograd.grad(predictive, params, retain_graph=True, allow_unused=True)
    g = torch.autograd.grad(decision_obj, params, allow_unused=True)
    common = [i for i, (x, y) in enumerate(zip(a, g)) if x is not None and y is not None]
    if common:
        dot = sum((a[i] * g[i]).sum() for i in common)
        aa = sum(a[i].square().sum() for i in common).clamp(min=1e-12)
        gg = sum(g[i].square().sum() for i in common).clamp(min=1e-12)
        cosine = float((dot / torch.sqrt(aa * gg)).detach())
        conflict = bool(dot.detach().item() < 0)
        alpha = (dot / gg).detach() if conflict else None
        beta = (dot / aa).detach() if conflict else None
    else:
        cosine = float('nan'); conflict = False; alpha = beta = None
    for i, p in enumerate(params):
        x, y = a[i], g[i]
        if x is not None and y is not None and conflict:
            p.grad = (x - alpha * y + y - beta * x).detach()
        elif x is not None and y is not None:
            p.grad = (x + y).detach()
        elif x is not None:
            p.grad = x.detach()
        elif y is not None:
            p.grad = y.detach()
    return {'gradient_cosine': cosine, 'gradient_conflict': conflict}

train_dl = dl(0, True); val_dl = dl(1, False)
outdir = Path(CFG['output_dir']); outdir.mkdir(parents=True, exist_ok=True)
STAGES = list(CFG['stage_epochs'])
assert STAGES == ['A_SSL', 'B_REGIME', 'D_FORECAST', 'E_PATH', 'F_EXPERTS', 'G_DECISION', 'H_JOINT']
resume_path = outdir / 'v8_btc_dollar_resume.pt'
best_path = outdir / 'v8_btc_dollar_H_JOINT_best.pt'
RESUME_SCHEMA = 4

source_info = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in paths]
contract = {
    'model': {
        'features': len(FEATURES), 'hidden': CFG['hidden'], 'experts': CFG['experts'],
        'topk': CFG['top_k'], 'layers': CFG['transformer_layers'],
        'gru_layers': CFG['gru_layers'], 'expert_width': CFG['expert_width'],
        'path_knots': CFG['path_knots'], 'path_rank': CFG['path_rank'],
        'mtf_timeframes': CFG['mtf_timeframes'], 'mtf_window': CFG['mtf_window'],
        'mtf_encoder_layers': CFG['mtf_encoder_layers'],
        'version': 'v8_BTC_DOLLAR_OVERHAUL_R2'
    },
    'horizons': H, 'lookback': lb, 'splits': [str(c) for c in cuts],
    'train_start': str(train_start), 'train_end': str(train_end),
    'sampling': {k: CFG[k] for k in ('train_years', 'recent_half_life_days', 'recent_uniform_floor', 'stress_boost', 'stress_vol_quantile', 'stress_cap')},
    'cache_fingerprint': CACHE_FINGERPRINT, 'sources': source_info,
    'med': med.to_dict(), 'iqr': iqr.to_dict(), 'ret_scale': ret_scale, 'rv_scale': rv_scale,
    'mtf_scalers': {k: {'med': v[0].to_dict(), 'iqr': v[1].to_dict()} for k, v in mtf_scalers.items()},
    'stage_epochs': CFG['stage_epochs'], 'stage_lrs': CFG['stage_lr_mult'],
    'stage_patience': CFG['stage_patience'], 'stage_max_val_steps': CFG['stage_max_val_steps'],
    'max_train_steps_per_epoch': CFG['max_train_steps_per_epoch'],
    'loss_weights': {k: CFG[k] for k in CFG if k.endswith('_weight')},
    'seed': SEED, 'optimizer': 'AdamW',
    'weight_decay': CFG['weight_decay'], 'dropout': CFG['dropout'], 'label_smoothing': CFG['label_smoothing'],
    'research_adverse_penalty': CFG['research_adverse_penalty'],
    'reference_cost_bps': CFG['reference_cost_bps'],
    'pcgrad': CFG['use_pcgrad_joint'], 'precision': 'bf16' if USE_BF16 else 'fp32',
    'checkpoint_every_steps': CFG['checkpoint_every_steps'], 'resume_schema': RESUME_SCHEMA,
}
contract_hash = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()

def select_stage_parameters(stage):
    temporal = ('input', 'pos_emb', 'in_drop', 'conv', 'encoder', 'longrange', 'ssm', 'backbone_fusion', 'mtf')
    forecast = ('mtf', 'direction', 'quantiles', 'rv', 'excursions', 'barrier', 'pattern', 'pattern_detail', 'jump', 'path', 'forecast_fusion', 'ctx_norm')
    if stage == 'A_SSL': prefixes = ('input', 'pos_emb', 'in_drop', 'conv', 'encoder', 'longrange', 'ssm', 'backbone_fusion', 'ssl_predict')
    elif stage == 'B_REGIME': prefixes = temporal + ('regime',)
    elif stage == 'D_FORECAST': prefixes = temporal + ('regime',) + forecast
    elif stage == 'E_PATH': prefixes = temporal + ('forecast_fusion', 'ctx_norm', 'path_distribution', 'path')
    elif stage == 'F_EXPERTS': prefixes = temporal + ('regime', 'router', 'experts', 'moe_scale', 'path_distribution') + forecast
    elif stage == 'G_DECISION': prefixes = ('decision', 'decision_quantiles')
    elif stage == 'H_JOINT': prefixes = tuple({s.split('.')[0] for s, _ in model.named_parameters()})
    else: raise ValueError(stage)
    for name, p in model.named_parameters():
        p.requires_grad = name.split('.')[0] in prefixes
    active = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert active > 0
    return active

STAGE_GRADIENT_HEADS = {
    'A_SSL': ('ssl_predict',),
    'B_REGIME': ('regime',),
    'D_FORECAST': ('direction', 'quantiles', 'rv', 'barrier', 'path'),
    'E_PATH': ('path_distribution', 'path'),
    'F_EXPERTS': ('router', 'direction', 'path_distribution'),
    'G_DECISION': ('decision', 'decision_quantiles'),
    'H_JOINT': ('direction', 'decision', 'path_distribution', 'path'),
}

def assert_stage_head_gradients(stage):
    if not CFG['diagnostic_gradient_assert']: return {}
    diagnostic = {}
    for head in STAGE_GRADIENT_HEADS[stage]:
        ps = [p for name, p in model.named_parameters() if name.startswith(head + '.')]
        if not ps: raise RuntimeError(f'{stage}: missing module {head}')
        if not any(p.requires_grad for p in ps): raise RuntimeError(f'{stage}: {head} frozen')
        grads = [p.grad for p in ps if p.grad is not None]
        if not grads: raise RuntimeError(f'{stage}: missing gradient to {head}')
        if any(not torch.isfinite(g).all().item() for g in grads): raise FloatingPointError(f'{stage}: nonfinite grad in {head}')
        magnitude = sum(float(g.detach().float().abs().sum()) for g in grads)
        if magnitude <= 0: raise RuntimeError(f'{stage}: zero grad in {head}')
        diagnostic[head] = magnitude
    return diagnostic

def stage_step(stage, b):
    x = b[0]
    if stage == 'A_SSL':
        o = model(x[:, :-1], use_experts=False, use_mtf=False)
        return F.smooth_l1_loss(o['ssl_prediction'], x[:, -1].clamp(-12, 12))
    has_experts = stage in ('F_EXPERTS', 'G_DECISION', 'H_JOINT')
    o = model(x, use_experts=has_experts, mtf=b[15], mtf_mask=b[16])
    if stage == 'B_REGIME': return regime_loss(o, b)
    if stage == 'D_FORECAST': return forecast_losses(o, b, include_paths=False, include_route=False)
    if stage == 'E_PATH': return path_distribution_nll(o, b[13]) + F.smooth_l1_loss(o['path'], b[10])
    if stage == 'F_EXPERTS': return forecast_losses(o, b, include_paths=True, include_route=True)
    if stage == 'G_DECISION': return action_loss(o, b)
    if stage == 'H_JOINT': return objective(o, b)
    raise ValueError(stage)

def atomic_save(obj, path):
    temp = path.with_suffix(path.suffix + '.tmp'); torch.save(obj, temp); temp.replace(path)

def rng_snapshot():
    s = {'torch': torch.get_rng_state(), 'numpy': np.random.get_state(), 'python': random.getstate()}
    if torch.cuda.is_available(): s['cuda'] = torch.cuda.get_rng_state_all()
    return s

def rng_restore(s):
    torch.set_rng_state(s['torch']); np.random.set_state(s['numpy']); random.setstate(s['python'])
    if torch.cuda.is_available() and 'cuda' in s: torch.cuda.set_rng_state_all(s['cuda'])

@torch.no_grad()
def expert_diagnostics(loader, limit=3):
    model.eval(); router_list = []; dispatch_list = []; ablation = []
    for i, pack in enumerate(loader):
        if i >= limit: break
        b = move_batch(pack)
        o = model(b[0], mtf=b[15], mtf_mask=b[16])
        loss = F.cross_entropy(o['direction'].reshape(-1, 3), b[2].reshape(-1)).item()
        router_list.append(o['router_probs'].mean(0).cpu().numpy())
        dispatch_list.append(o['dispatch'].mean(0).cpu().numpy())
        if CFG['validate_expert_ablation']:
            deltas = []
            for idx in range(CFG['experts']):
                oa = model(b[0], mtf=b[15], mtf_mask=b[16], ablated_expert=idx)
                deltas.append(F.cross_entropy(oa['direction'].reshape(-1, 3), b[2].reshape(-1)).item() - loss)
            ablation.append(deltas)
    model.train()
    if not router_list: return {'available': False}
    return {
        'available': True, 'batches': len(router_list),
        'expert_router_prob': np.mean(router_list, axis=0).round(5).tolist(),
        'expert_actual_dispatch': np.mean(dispatch_list, axis=0).round(5).tolist(),
        'direction_loss_increase_without_each_expert': np.mean(ablation, axis=0).round(6).tolist() if ablation else None,
    }

def verify_real_training_batches():
    train_pack = next(iter(train_dl)); validation_pack = next(iter(val_dl))
    saved_rng = rng_snapshot(); was_training = model.training
    previous_requires = {name: p.requires_grad for name, p in model.named_parameters()}
    try:
        for _, p in model.named_parameters(): p.requires_grad = True
        model.train(); model.zero_grad(set_to_none=True)
        bt = move_batch(train_pack)
        with amp_context():
            o = model(bt[0], mtf=bt[15], mtf_mask=bt[16])
            pred_loss = forecast_losses(o, bt); dec_loss = action_loss(o, bt)
            train_loss = pred_loss + dec_loss
        if not torch.isfinite(train_loss): raise FloatingPointError('Nonfinite preflight train loss')
        with amp_context(): diag = joint_pcgrad_backward(pred_loss, dec_loss)
        model.eval()
        with torch.no_grad():
            bv = move_batch(validation_pack)
            with amp_context(): val_loss = objective(model(bv[0], mtf=bv[15], mtf_mask=bv[16]), bv)
        if not torch.isfinite(val_loss): raise FloatingPointError('Nonfinite preflight val loss')
        print('REAL-DATA PREFLIGHT PASS', {
            'train_batch': len(bt[0]), 'validation_batch': len(bv[0]),
            'train_loss': float(train_loss.detach()), 'validation_loss': float(val_loss.detach()),
            'gradient_cosine': diag['gradient_cosine']
        })
    finally:
        model.zero_grad(set_to_none=True)
        for name, p in model.named_parameters(): p.requires_grad = previous_requires[name]
        model.train(was_training); rng_restore(saved_rng)

verify_real_training_batches()

def checked_resume_payload(cp):
    if cp.get('resume_schema') != RESUME_SCHEMA: raise RuntimeError('Unsupported checkpoint schema')
    if cp.get('contract_hash') != contract_hash: raise RuntimeError('Contract mismatch')
    return cp

history = []; stage_index = 0; epoch_in_stage = 0; stage_best = float('inf'); bad = 0; checkpoint = None
batch_in_epoch = 0; partial_sum = 0.; partial_seen = 0
partial_cosine_sum = 0.; partial_cosine_count = 0; partial_conflicts = 0
if resume_path.exists():
    checkpoint = checked_resume_payload(torch.load(resume_path, map_location='cpu', weights_only=False))
    model.load_state_dict(checkpoint['model']); stage_index = checkpoint['stage_index']
    epoch_in_stage = checkpoint['epoch_in_stage']; stage_best = checkpoint['stage_best']
    bad = checkpoint['bad']; history = checkpoint['history']; rng_restore(checkpoint['rng'])
    batch_in_epoch = checkpoint['batch_in_epoch']; partial_sum = checkpoint['partial_sum']; partial_seen = checkpoint['partial_seen']
    partial_cosine_sum = checkpoint['partial_cosine_sum']; partial_cosine_count = checkpoint['partial_cosine_count']
    partial_conflicts = checkpoint['partial_conflicts']
    print('Resuming:', STAGES[stage_index] if stage_index < len(STAGES) else 'COMPLETE', 'epoch:', epoch_in_stage, 'batches:', batch_in_epoch)

for si in range(stage_index, len(STAGES)):
    stage = STAGES[si]; active = select_stage_parameters(stage)
    lr = CFG['lr'] * CFG['stage_lr_mult'][stage]
    decay_p = [p for n, p in model.named_parameters() if p.requires_grad and p.ndim >= 2 and not n.endswith('pos_emb') and 'tokens' not in n]
    nodecay_p = [p for n, p in model.named_parameters() if p.requires_grad and not (p.ndim >= 2 and not n.endswith('pos_emb') and 'tokens' not in n)]
    opt = torch.optim.AdamW([
        {'params': decay_p, 'weight_decay': float(CFG['weight_decay'])},
        {'params': nodecay_p, 'weight_decay': 0.0}
    ], lr=lr, fused=(DEVICE.type == 'cuda'))
    steps_per_epoch = min(math.ceil(len(train_dl.dataset) / CFG['batch_size']), int(CFG['max_train_steps_per_epoch'])) if CFG['max_train_steps_per_epoch'] else math.ceil(len(train_dl.dataset) / CFG['batch_size'])
    total_stage_steps = max(1, int(CFG['stage_epochs'][stage]) * steps_per_epoch)
    warmup_steps = min(100, max(1, total_stage_steps // 10))
    def _lr_lambda(step_idx):
        if step_idx < warmup_steps:
            return 0.1 + 0.9 * (step_idx + 1) / warmup_steps
        prog = min(1.0, (step_idx - warmup_steps) / max(1, total_stage_steps - warmup_steps))
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=_lr_lambda)
    if si == stage_index and (epoch_in_stage > 0 or batch_in_epoch > 0):
        opt.load_state_dict(checkpoint['optimizer'])
        if 'scheduler' in checkpoint and checkpoint['scheduler'] is not None:
            scheduler.load_state_dict(checkpoint['scheduler'])
    else:
        epoch_in_stage = 0; stage_best = float('inf'); bad = 0; batch_in_epoch = 0; partial_sum = 0.; partial_seen = 0
        partial_cosine_sum = 0.; partial_cosine_count = 0; partial_conflicts = 0
    stage_best_path = outdir / f'v8_btc_dollar_{stage}_best.pt'
    print('Stage', stage, 'trainable', active, '/', PARAMETERS, 'lr', lr)
    for epoch in range(epoch_in_stage, int(CFG['stage_epochs'][stage])):
        if bad >= CFG['stage_patience'][stage]: break
        model.train(); train_sum = partial_sum; train_seen = partial_seen
        cosine_sum = partial_cosine_sum; cosine_count = partial_cosine_count; conflicts = partial_conflicts
        offset = batch_in_epoch if (si == stage_index and epoch == epoch_in_stage) else 0
        train_dl.sampler.set_epoch('btc', stage, epoch, offset)
        for step, pack in enumerate(train_dl, start=offset):
            b = move_batch(pack)
            opt.zero_grad(set_to_none=True)
            with amp_context():
                if stage == 'H_JOINT' and CFG['use_pcgrad_joint']:
                    o = model(b[0], mtf=b[15], mtf_mask=b[16])
                    pred_loss = forecast_losses(o, b); dec_loss = action_loss(o, b)
                    loss = pred_loss + dec_loss
                else:
                    loss = stage_step(stage, b)
            if not torch.isfinite(loss): raise FloatingPointError(f'{stage} nonfinite loss')
            if stage == 'H_JOINT' and CFG['use_pcgrad_joint']:
                with amp_context(): diag = joint_pcgrad_backward(pred_loss, dec_loss)
                if np.isfinite(diag['gradient_cosine']):
                    cosine_sum += float(diag['gradient_cosine']); cosine_count += 1
                conflicts += int(diag['gradient_conflict'])
            else:
                loss.backward()
            if step == 0: assert_stage_head_gradients(stage)
            norm = nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.)
            if not torch.isfinite(norm): raise FloatingPointError(f'{stage} nonfinite gradient')
            opt.step(); scheduler.step(); train_sum += float(loss.detach()) * len(b[0]); train_seen += len(b[0])
            if CFG['checkpoint_every_steps'] and (step + 1) % CFG['checkpoint_every_steps'] == 0:
                atomic_save({
                    'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(), 'contract_hash': contract_hash,
                    'stage_index': si, 'epoch_in_stage': epoch, 'batch_in_epoch': step + 1,
                    'partial_sum': train_sum, 'partial_seen': train_seen, 'stage_best': stage_best,
                    'bad': bad, 'history': history, 'rng': rng_snapshot(), 'resume_schema': RESUME_SCHEMA,
                    'partial_cosine_sum': cosine_sum, 'partial_cosine_count': cosine_count, 'partial_conflicts': conflicts
                }, resume_path)
            if CFG['max_train_steps_per_epoch'] is not None and step + 1 >= CFG['max_train_steps_per_epoch']:
                break
        model.eval(); val_sum = 0.; val_seen = 0; val_dec_sum = 0.; val_pred_sum = 0.
        with torch.no_grad():
            for step, pack in enumerate(val_dl):
                b = move_batch(pack)
                with amp_context():
                    if stage == 'H_JOINT':
                        ov = model(b[0], mtf=b[15], mtf_mask=b[16])
                        p_l = forecast_losses(ov, b); d_l = action_loss(ov, b)
                        loss = p_l + d_l
                        val_pred_sum += float(p_l) * len(b[0]); val_dec_sum += float(d_l) * len(b[0])
                    else:
                        loss = stage_step(stage, b)
                if not torch.isfinite(loss): raise FloatingPointError(f'{stage} nonfinite validation')
                val_sum += float(loss) * len(b[0]); val_seen += len(b[0])
                if CFG['stage_max_val_steps'] is not None and step + 1 >= CFG['stage_max_val_steps']:
                    break
        score = (val_pred_sum + 2.0 * val_dec_sum) / val_seen if stage == 'H_JOINT' else (val_sum / val_seen)
        entry = {
            'stage': stage, 'epoch': epoch + 1, 'train_loss': train_sum / train_seen,
            'validation_loss': score, 'active_parameters': active, 'train_seen': train_seen,
            'validation_seen': val_seen, 'lr': lr,
            'gradient_cosine': cosine_sum / cosine_count if cosine_count else None,
            'gradient_conflict_steps': conflicts
        }
        if stage == 'H_JOINT':
            entry['val_pred_loss'] = val_pred_sum / val_seen
            entry['val_decision_loss'] = val_dec_sum / val_seen
        if stage in ('F_EXPERTS', 'H_JOINT'):
            entry['expert_diagnostics'] = expert_diagnostics(val_dl, CFG['expert_val_batches'])
        history.append(entry); print(entry)
        if score < stage_best - 1e-6:
            stage_best = score; bad = 0
            atomic_save({
                'model': model.state_dict(), 'contract_hash': contract_hash,
                'stage': stage, 'epoch': epoch + 1, 'validation_loss': score, 'resume_schema': RESUME_SCHEMA
            }, stage_best_path)
        else:
            bad += 1
        atomic_save({
            'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(), 'contract_hash': contract_hash,
            'stage_index': si, 'epoch_in_stage': epoch + 1, 'stage_best': stage_best, 'bad': bad,
            'history': history, 'rng': rng_snapshot(), 'batch_in_epoch': 0, 'partial_sum': 0., 'partial_seen': 0,
            'partial_cosine_sum': 0., 'partial_cosine_count': 0, 'partial_conflicts': 0, 'resume_schema': RESUME_SCHEMA
        }, resume_path)
        batch_in_epoch = 0; partial_sum = 0.; partial_seen = 0
        partial_cosine_sum = 0.; partial_cosine_count = 0; partial_conflicts = 0
        if bad >= CFG['stage_patience'][stage]: break

    best_stage = torch.load(stage_best_path, map_location='cpu', weights_only=False)
    if best_stage.get('contract_hash') != contract_hash or best_stage.get('stage') != stage:
        raise RuntimeError(f'Mismatched stage best checkpoint: {stage_best_path}')
    model.load_state_dict(best_stage['model'])
    atomic_save({
        'model': model.state_dict(), 'optimizer': None, 'contract_hash': contract_hash,
        'stage_index': si + 1, 'epoch_in_stage': 0, 'stage_best': float('inf'), 'bad': 0,
        'history': history, 'rng': rng_snapshot(), 'batch_in_epoch': 0, 'partial_sum': 0., 'partial_seen': 0,
        'partial_cosine_sum': 0., 'partial_cosine_count': 0, 'partial_conflicts': 0, 'resume_schema': RESUME_SCHEMA
    }, resume_path)
    epoch_in_stage = 0; batch_in_epoch = 0; partial_sum = 0.; partial_seen = 0
    partial_cosine_sum = 0.; partial_cosine_count = 0; partial_conflicts = 0

final_best = torch.load(best_path, map_location='cpu', weights_only=False)
assert final_best.get('contract_hash') == contract_hash and final_best.get('stage') == 'H_JOINT'
model.load_state_dict(final_best['model']); model.eval()
print('7-stage BTC-only Dollar-Bar training complete!')

## 4. Calibration, Quality Diagnostics & Sealed Evaluation
- Temperature calibration runs streaming over the separate Calibration partition (fold 2).
- Quality diagnostics evaluate on the independent late-validation report cohort (`validation_role='report'`).
- Sealed OOS (fold 3) remains locked (`run_sealed_test=False`) until validation gates pass.


In [ ]:
# Cell 13 — Streaming calibration, independent validation diagnostics, and locked sealed OOS
from scipy.special import softmax

@torch.no_grad()
def _collect_calibration_logits():
    logits_list = []; targets_list = []
    model.eval()
    for pack in dl(2, False):
        b = move_batch(pack)
        with amp_context():
            logits = model(b[0], mtf=b[15], mtf_mask=b[16])['decision'].float()
        mask = b[17].bool()
        if not mask.any(): continue
        logits_list.append(logits[mask].cpu())
        targets_list.append(b[6][mask].long().cpu())
    if not logits_list: raise RuntimeError('Calibration set has no examples')
    return torch.cat(logits_list, dim=0), torch.cat(targets_list, dim=0)

def streaming_temperature_fit():
    cal_logits, cal_targets = _collect_calibration_logits()
    seen = int(len(cal_targets))
    log_temp = torch.zeros(1, requires_grad=True)
    bias_raw = torch.zeros(3, requires_grad=True)
    opt_cal = torch.optim.Adam([log_temp, bias_raw], lr=0.05)
    for _ in range(250):
        opt_cal.zero_grad()
        t_val = torch.exp(log_temp).clamp(0.4, 4.0)
        bias_c = bias_raw - bias_raw.mean()
        loss = F.cross_entropy(cal_logits / t_val + bias_c, cal_targets) + 1e-4 * bias_c.square().sum()
        loss.backward()
        opt_cal.step()
    with torch.no_grad():
        t_final = float(torch.exp(log_temp).clamp(0.4, 4.0).item())
        b_final = (bias_raw - bias_raw.mean()).numpy().astype(np.float64)
        nll = float(F.cross_entropy(cal_logits / t_final + torch.from_numpy(b_final).float(), cal_targets).item())
    return t_final, b_final, seen, nll

TEMPERATURE, CAL_BIAS_NP, N_CAL, CAL_LOGLOSS = streaming_temperature_fit()
CAL_BIAS_T = torch.tensor(CAL_BIAS_NP, device=DEVICE, dtype=torch.float32)
(outdir / 'calibration.json').write_text(json.dumps({
    'temperature': TEMPERATURE, 'class_bias': CAL_BIAS_NP.tolist(),
    'n_calibration': N_CAL, 'multiclass_log_loss': CAL_LOGLOSS,
    'contract_hash': contract_hash, 'frozen_checkpoint': str(best_path)
}, indent=2))
print('Decision temperature:', TEMPERATURE, 'bias:', np.round(CAL_BIAS_NP, 4).tolist(), 'calibration_n:', N_CAL)

@torch.no_grad()
def training_quality_diagnostics():
    model.eval(); nh = len(H); n = 0
    dlog = np.zeros(nh); dcorr = np.zeros(nh); dmajor = np.zeros(nh)
    rmae = np.zeros(nh); zero_mae = np.zeros(nh); q10 = np.zeros(nh); q90 = np.zeros(nh)
    confusion = np.zeros((3, 3), dtype=np.int64); d_brier = 0.; d_ll = 0.; d_count = 0
    hi_stats = {0.45: [0, 0, 0, 0], 0.50: [0, 0, 0, 0]}  # [s_hit, s_n, l_hit, l_n]
    dispatch = []; routes = []
    for pack in dl(1, False, validation_role='report'):
        b = move_batch(pack)
        with amp_context(): o = model(b[0], mtf=b[15], mtf_mask=b[16])
        B = len(b[0]); n += B
        direction = o['direction'].float(); real = b[2]
        dlog += F.cross_entropy(direction.reshape(-1, 3), real.reshape(-1), reduction='none').reshape(B, nh).sum(0).cpu().numpy()
        pred = direction.argmax(-1)
        dcorr += (pred == real).sum(0).cpu().numpy()
        for j in range(nh):
            dmajor[j] += int(torch.bincount(real[:, j], minlength=3).max())
        med_q = o['quantiles'].float()[..., 1]; ret = b[1].float()
        rmae += (med_q - ret).abs().sum(0).cpu().numpy(); zero_mae += ret.abs().sum(0).cpu().numpy()
        q10 += (ret <= o['quantiles'][..., 0]).sum(0).cpu().numpy()
        q90 += (ret <= o['quantiles'][..., 2]).sum(0).cpu().numpy()
        valid = b[17].bool()
        if valid.any():
            logits = o['decision'].float()[valid] / TEMPERATURE + CAL_BIAS_T
            label = b[6][valid].long(); probs = F.softmax(logits, dim=-1); pred_d = probs.argmax(-1)
            p_np = probs.cpu().numpy(); y_np = label.cpu().numpy()
            for actual, predicted in zip(y_np.tolist(), pred_d.cpu().tolist()):
                confusion[actual, predicted] += 1
            for th in hi_stats:
                ms = (p_np[:, 0] >= th) & ((p_np[:, 0] - p_np[:, 2]) >= 0.05)
                ml = (p_np[:, 2] >= th) & ((p_np[:, 2] - p_np[:, 0]) >= 0.05)
                hi_stats[th][0] += int((y_np[ms] == 0).sum()); hi_stats[th][1] += int(ms.sum())
                hi_stats[th][2] += int((y_np[ml] == 2).sum()); hi_stats[th][3] += int(ml.sum())
            d_ll += float(F.cross_entropy(logits, label, reduction='sum'))
            d_brier += float(((probs - F.one_hot(label, 3).float()) ** 2).sum())
            d_count += len(label)
        dispatch.append((o['dispatch'].float().sum(0).cpu().numpy(), B))
        routes.append((o['router_probs'].float().sum(0).cpu().numpy(), B))
    c = confusion; support = c.sum(1); predicted = c.sum(0)
    precision = np.divide(c.diagonal(), predicted, out=np.zeros(3), where=predicted > 0)
    recall = np.divide(c.diagonal(), support, out=np.zeros(3), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(3), where=(precision + recall) > 0)
    prior = support / max(1, d_count)
    return {
        'cohort': 'independent late BTC Dollar-Bar validation report',
        'samples': n, 'horizons_dollar_bars': H,
        'direction_ce': (dlog / n).tolist(), 'direction_accuracy': (dcorr / n).tolist(),
        'batch_majority_accuracy_upper_bound': (dmajor / n).tolist(),
        'median_mae': (rmae / n).tolist(), 'zero_return_mae': (zero_mae / n).tolist(),
        'q10_coverage': (q10 / n).tolist(), 'q90_coverage': (q90 / n).tolist(),
        'decision_samples': d_count, 'decision_confusion_rows_truth_cols_pred': c.tolist(),
        'decision_macro_f1': float(f1[support > 0].mean()) if any(support > 0) else None,
        'decision_log_loss': d_ll / max(1, d_count), 'decision_brier': d_brier / max(1, d_count),
        'decision_class_prior_brier': float(1. - np.square(prior).sum()) if d_count else None,
        'high_conf_precision': {
            str(th): {
                'short_n': v[1], 'short_prec': (v[0] / v[1]) if v[1] else None,
                'long_n': v[3], 'long_prec': (v[2] / v[3]) if v[3] else None,
            } for th, v in hi_stats.items()
        },
        'expert_dispatch_frequency': (sum(v for v, _ in dispatch) / n).tolist(),
        'expert_router_probability': (sum(v for v, _ in routes) / n).tolist()
    }

training_quality_report = training_quality_diagnostics()
(outdir / 'training_quality_diagnostics.json').write_text(json.dumps(training_quality_report, indent=2))
print('TRAINING QUALITY DIAGNOSTICS:', json.dumps(training_quality_report, indent=2))

@torch.no_grad()
def evaluate_sealed_streaming():
    if not CFG['run_sealed_test']:
        raise RuntimeError('Sealed OOS must be explicitly enabled after model freeze')
    model.eval(); n = 0; loss_sum = 0.; brier_sum = 0.; correct = np.zeros(3); true_count = np.zeros(3)
    pred_count = np.zeros(3); trade_count = 0; signals_raw = []
    ds = LazyFold(3); offset = 0
    for pack in dl(3, False):
        b = move_batch(pack); o = model(b[0], mtf=b[15], mtf_mask=b[16])
        decision_ok = b[17].bool()
        if not decision_ok.any():
            offset += len(b[0]); continue
        logits = o['decision'][decision_ok].float() / TEMPERATURE + CAL_BIAS_T; prob = F.softmax(logits, dim=-1)
        pred = prob.argmax(-1); true = b[6][decision_ok]
        loss_sum += float(F.cross_entropy(logits, true, reduction='sum'))
        brier_sum += float(((prob - F.one_hot(true, 3).float()) ** 2).sum())
        trade_count += int((pred != 1).sum())
        for k in range(3):
            true_count[k] += int((true == k).sum()); pred_count[k] += int((pred == k).sum())
            correct[k] += int(((pred == k) & (true == k)).sum())
        if len(signals_raw) < 5:
            q = path_forecast_quantiles(o).detach().cpu().numpy()
            valid_pos = torch.nonzero(decision_ok, as_tuple=True)[0].cpu().tolist()
            for j in valid_pos[:max(0, 5 - len(signals_raw))]:
                signals_raw.append((ds.ref(offset + j), {k: v[j].detach().cpu().numpy() for k, v in o.items()}, q[j]))
        offset += len(b[0]); n += len(true)
    recall = np.divide(correct, true_count, out=np.zeros_like(correct), where=true_count > 0)
    return ({
        'samples': n, 'decision_log_loss': loss_sum / n, 'brier_score': brier_sum / n,
        'balanced_accuracy': float(recall[true_count > 0].mean()),
        'true_class_counts': true_count.astype(int).tolist(),
        'pred_class_counts': pred_count.astype(int).tolist(),
        'n_nonwait_signals': trade_count, 'sealed_evaluation_is_one_time': True
    }, signals_raw)

metrics = None; sealed_examples = []
if CFG['run_sealed_test']:
    metrics, sealed_examples = evaluate_sealed_streaming()
    print('SEALED OOS:', json.dumps(metrics, indent=2))
else:
    print('Sealed OOS locked. Calibration and validation reports saved.')

In [ ]:
# Cell 14 — Research signal contract (no broker execution)
CLASSES = ['SHORT', 'WAIT', 'LONG']
REGIME_NAMES = ['trend_persistence', 'high_volatility', 'transition', 'liquidity_cost',
                'past_jump', 'market_complexity', 'cross_asset_coupling']
PATTERN_TYPES = ['breakout', 'false_breakout', 'reversal', 'trend_continuation']

def make_research_signal(ref, row, scenario_quantiles):
    asset, asof = ref
    probs = softmax(row['decision'] / TEMPERATURE + CAL_BIAS_NP, axis=-1)
    patterns = np.concatenate([row['pattern'][:, None], row['pattern_detail']], axis=-1)
    return {
        'schema': 'manticore.quant_signal.v3.0.dollar_bar',
        'asset': asset, 'asof_utc': asof, 'bar_type': f"dollar_{int(CFG['dollar_bar_target_usd'])}",
        'timeframes': ['DOLLAR'] + TF_NAMES, 'decision': CLASSES[int(probs.argmax())],
        'decision_probabilities': dict(zip(CLASSES, np.round(probs, 6).tolist())),
        'forecast_horizons_bars': H, 'future_path_knots_bars': CFG['path_knots'],
        'forecast': {
            'direction_probabilities': softmax(row['direction'], axis=-1).round(6).tolist(),
            'log_return_quantiles_p10_p50_p90': row['quantiles'].round(7).tolist(),
            'future_realized_variance': row['rv'].round(10).tolist(),
            'first_touch_probs_by_horizon': softmax(row['barrier'], axis=-1).round(6).tolist(),
            'pattern_names': PATTERN_TYPES,
            'pattern_event_probabilities': (1 / (1 + np.exp(-patterns))).round(6).tolist(),
            'jump_event_probabilities': (1 / (1 + np.exp(-row['jump']))).round(6).tolist(),
            'correlated_path_log_return_quantiles': np.asarray(scenario_quantiles).round(7).tolist(),
        },
        'market_state': {
            'regime_names': REGIME_NAMES,
            'regime_probabilities': (1 / (1 + np.exp(-row['regime']))).round(6).tolist(),
            'expert_weights': row['gate'].round(5).tolist(),
            'mtf_attention_weights': dict(zip(TF_NAMES, row['mtf_attention'].round(5).tolist())),
            'actual_routed_experts': np.where(row['dispatch'] > 0)[0].tolist()
        },
        'decision_action_log_utility_quantiles': row['action_quantiles'].round(7).tolist(),
        'uncertainty': {'decision_entropy': float(-(probs * np.log(np.maximum(probs, 1e-12))).sum())},
        'execution_authorized': False
    }

signals = [make_research_signal(ref, row, q) for ref, row, q in sealed_examples]
if signals: print(json.dumps(signals[0], indent=2))

In [ ]:
# Cell 15 — Save final BTC Dollar-Bar model and reports
outdir = Path(CFG['output_dir']); outdir.mkdir(parents=True, exist_ok=True)
atomic_save({
    'model_state': model.state_dict(),
    'model_config': {
        'features': len(FEATURES), 'hidden': CFG['hidden'], 'nexperts': CFG['experts'],
        'topk': CFG['top_k'], 'nh': len(H), 'layers': CFG['transformer_layers'],
        'heads': CFG['attention_heads'], 'gru_layers': CFG['gru_layers'],
        'expert_width': CFG['expert_width'], 'path_knots': CFG['path_knots'], 'path_rank': CFG['path_rank']
    },
    'features': FEATURES, 'horizons': H, 'dollar_bar_target_usd': CFG['dollar_bar_target_usd'],
    'med': med.to_dict(), 'iqr': iqr.to_dict(), 'ret_scale': ret_scale, 'rv_scale': rv_scale,
    'contract_hash': contract_hash,
    'mtf_scalers': {k: {'med': v[0].to_dict(), 'iqr': v[1].to_dict()} for k, v in mtf_scalers.items()},
    'calibration_temperature': TEMPERATURE, 'calibration_bias': CAL_BIAS_NP.tolist(), 'research_only': True, 'no_broker_execution': True,
    'training_mode': 'btc_only_dollar_bars_7stage', 'stage_epochs': CFG['stage_epochs'],
    'target_asset': CFG['target_asset']
}, outdir / f'ananta_v8_btc_dollar_{SCALE_PROFILE.lower()}.pt')
if metrics is not None: (outdir / 'sealed_oos_metrics.json').write_text(json.dumps(metrics, indent=2))
(outdir / 'research_signals.jsonl').write_text('\n'.join(json.dumps(s) for s in signals) + '\n')
(outdir / 'training_history.json').write_text(json.dumps({'btc_dollar': history}, indent=2))
(outdir / 'data_coverage.json').write_text(json.dumps(coverage, indent=2))
print('Saved research artifacts:', outdir)

In [ ]:
# Cell 16 — Save Validation-Report Predictions & Run 3-Scale Ensemble Non-Overlapping Backtest
d_full = load_asset_file(btc_path)
ds_rep = LazyFold(1, validation_role='report')
loader_rep = dl(1, False, validation_role='report')
probs_list = []; uq_list = []
model.eval()
with torch.no_grad():
    for pack in loader_rep:
        b = move_batch(pack)
        with amp_context():
            out_rep = model(b[0], mtf=b[15], mtf_mask=b[16])
            logits = out_rep['decision'].float()
            uq = out_rep['action_quantiles'].float()[..., 1]
        probs_list.append(F.softmax(logits / TEMPERATURE + CAL_BIAS_T, dim=-1).cpu().numpy())
        uq_list.append(uq.cpu().numpy())
probs_all = np.concatenate(probs_list, axis=0)
uq_all = np.concatenate(uq_list, axis=0)
rep_times = pd.to_datetime(np.asarray(ds_rep.T[ds_rep.ids]), utc=True)
_lr_full = np.log(d_full['close']).diff()
_vol_full = _lr_full.rolling(48, min_periods=24).std().clip(lower=1e-8)
closes = d_full.loc[rep_times, 'close'].to_numpy(dtype=np.float64)
highs = d_full.loc[rep_times, 'high'].to_numpy(dtype=np.float64)
lows = d_full.loc[rep_times, 'low'].to_numpy(dtype=np.float64)
local_vols = _vol_full.loc[rep_times].fillna(1e-4).to_numpy(dtype=np.float64)

pred_df = pd.DataFrame({
    'timestamp': rep_times, 'close': closes, 'high': highs, 'low': lows, 'local_vol': local_vols,
    'p_short': probs_all[:, 0], 'p_wait': probs_all[:, 1], 'p_long': probs_all[:, 2],
    'u_short': uq_all[:, 0], 'u_wait': uq_all[:, 1], 'u_long': uq_all[:, 2]
})
pred_path = outdir / f'val_report_preds_{SCALE_PROFILE.lower()}.parquet'
pred_df.to_parquet(pred_path, index=False)
print('Saved validation report predictions:', pred_path, len(pred_df))

def run_nonoverlapping_backtest(df_bt, horizon_bars=48, fee_rt=0.0008, margin=0.05, label='Single'):
    c = df_bt['close'].to_numpy(dtype=np.float64)
    ps = df_bt['p_short'].to_numpy(dtype=np.float64)
    pl = df_bt['p_long'].to_numpy(dtype=np.float64)
    ul = df_bt['u_long'].to_numpy(dtype=np.float64) if 'u_long' in df_bt else np.ones_like(pl)
    us = df_bt['u_short'].to_numpy(dtype=np.float64) if 'u_short' in df_bt else np.ones_like(ps)
    for thresh in (0.40, 0.45, 0.50, 0.55, 0.60):
        eq = 1.0; peak = 1.0; max_dd = 0.0; trades = []; l_tr = []; s_tr = []; i = 0; n_rep = len(c)
        while i + horizon_bars < n_rep:
            if pl[i] >= thresh and (pl[i] - ps[i]) >= margin and ul[i] >= us[i]:
                ret = (c[i + horizon_bars] / c[i]) - 1.0 - fee_rt
                trades.append(ret); l_tr.append(ret); eq *= (1.0 + ret)
                peak = max(peak, eq); max_dd = min(max_dd, (eq - peak) / peak)
                i += horizon_bars
            elif ps[i] >= thresh and (ps[i] - pl[i]) >= margin and us[i] >= ul[i]:
                ret = 1.0 - (c[i + horizon_bars] / c[i]) - fee_rt
                trades.append(ret); s_tr.append(ret); eq *= (1.0 + ret)
                peak = max(peak, eq); max_dd = min(max_dd, (eq - peak) / peak)
                i += horizon_bars
            else:
                i += 1
        arr = np.asarray(trades, dtype=np.float64)
        la = np.asarray(l_tr, dtype=np.float64); sa = np.asarray(s_tr, dtype=np.float64)
        win = float((arr > 0).mean()) if len(arr) else 0.0
        l_win = float((la > 0).mean()) if len(la) else 0.0
        s_win = float((sa > 0).mean()) if len(sa) else 0.0
        pf = float(arr[arr > 0].sum() / max(abs(arr[arr < 0].sum()), 1e-12)) if len(arr) else 0.0
        sh = float(arr.mean() / max(arr.std(), 1e-12)) if len(arr) > 1 else 0.0
        print(f'[{label}|h={horizon_bars}] Thresh={thresh:.2f} | N={len(arr)} (L={len(la)}/{l_win:.2f}, S={len(sa)}/{s_win:.2f}) | win={win:.3f} | PF={pf:.2f} | sr={sh:.2f} | ret={(eq-1)*100:.2f}% | dd={max_dd*100:.2f}%')

def run_triple_barrier_backtest(df_bt, horizon_bars=48, fee_rt=0.0008, margin=0.05, label='Single'):
    if not {'high', 'low', 'local_vol'}.issubset(df_bt.columns):
        return
    c = df_bt['close'].to_numpy(dtype=np.float64)
    hi = df_bt['high'].to_numpy(dtype=np.float64)
    lo = df_bt['low'].to_numpy(dtype=np.float64)
    lv = df_bt['local_vol'].to_numpy(dtype=np.float64)
    ps = df_bt['p_short'].to_numpy(dtype=np.float64)
    pl = df_bt['p_long'].to_numpy(dtype=np.float64)
    ul = df_bt['u_long'].to_numpy(dtype=np.float64) if 'u_long' in df_bt else np.ones_like(pl)
    us = df_bt['u_short'].to_numpy(dtype=np.float64) if 'u_short' in df_bt else np.ones_like(ps)
    mult = float(CFG['barrier_vol_multiplier'])
    min_mv = float(CFG['neutral_move_bps']) * 1e-4
    for min_edge in (0.0, 0.0025):
      for thresh in (0.45, 0.50, 0.55):
        eq = 1.0; peak = 1.0; max_dd = 0.0; trades = []; l_tr = []; s_tr = []; i = 0; n_rep = len(c)
        while i + horizon_bars < n_rep:
            up_r = max(mult * lv[i] * np.sqrt(horizon_bars), min_mv)
            go_l = up_r >= min_edge and pl[i] >= thresh and (pl[i] - ps[i]) >= margin and ul[i] >= us[i]
            go_s = up_r >= min_edge and ps[i] >= thresh and (ps[i] - pl[i]) >= margin and us[i] >= ul[i]
            if not (go_l or go_s):
                i += 1; continue
            tp = c[i] * np.exp(up_r) if go_l else c[i] * np.exp(-up_r)
            sl = c[i] * np.exp(-up_r) if go_l else c[i] * np.exp(up_r)
            k_exit = horizon_bars; exit_p = c[i + horizon_bars]
            for k in range(1, horizon_bars + 1):
                if go_l:
                    hit_sl = lo[i + k] <= sl; hit_tp = hi[i + k] >= tp
                    if hit_sl: k_exit = k; exit_p = sl; break
                    if hit_tp: k_exit = k; exit_p = tp; break
                else:
                    hit_sl = hi[i + k] >= sl; hit_tp = lo[i + k] <= tp
                    if hit_sl: k_exit = k; exit_p = sl; break
                    if hit_tp: k_exit = k; exit_p = tp; break
            ret = ((exit_p / c[i]) - 1.0 if go_l else 1.0 - (exit_p / c[i])) - fee_rt
            trades.append(ret); (l_tr if go_l else s_tr).append(ret)
            eq *= (1.0 + ret); peak = max(peak, eq); max_dd = min(max_dd, (eq - peak) / peak)
            i += k_exit
        arr = np.asarray(trades, dtype=np.float64)
        la = np.asarray(l_tr, dtype=np.float64); sa = np.asarray(s_tr, dtype=np.float64)
        win = float((arr > 0).mean()) if len(arr) else 0.0
        pf = float(arr[arr > 0].sum() / max(abs(arr[arr < 0].sum()), 1e-12)) if len(arr) else 0.0
        sh = float(arr.mean() / max(arr.std(), 1e-12)) if len(arr) > 1 else 0.0
        l_w = float((la > 0).mean()) if len(la) else 0.0; s_w = float((sa > 0).mean()) if len(sa) else 0.0
        print(f'[TB|{label}|h={horizon_bars}|tp>={min_edge*1e4:.0f}bp] T={thresh:.2f} | N={len(arr)} (L={len(la)}/{l_w:.2f}, S={len(sa)}/{s_w:.2f}) | win={win:.3f} | PF={pf:.2f} | sr={sh:.2f} | ret={(eq-1)*100:.2f}% | dd={max_dd*100:.2f}%')

RUN_BACKTEST_SWEEP = True
if RUN_BACKTEST_SWEEP:
    for hb in (H[-2], H[-1]):
        run_nonoverlapping_backtest(pred_df, horizon_bars=hb, label=SCALE_PROFILE)
    run_triple_barrier_backtest(pred_df, horizon_bars=H[-1], label=SCALE_PROFILE)

base_ckpt = Path('/content/drive/MyDrive/HERMES_Colab/ai_training/manticore_v70_trend_specialist_v2')
ens_files = {
    k: base_ckpt / f'v8_fullpower_{k.lower()}_checkpoints' / f'val_report_preds_{k.lower()}.parquet'
    for k in ('100K', '250K', '500K', '1500K', '3000K')
}
ready = {k: p for k, p in ens_files.items() if p.exists()}
if len(ready) >= 2:
    keys = list(ready.keys())
    base_k = '500K' if '500K' in ready else keys[0]
    pq_map = {k: pd.read_parquet(ready[k]).sort_values('timestamp') for k in keys}
    has_u = all({'u_short', 'u_wait', 'u_long'}.issubset(pq_map[k].columns) for k in keys)
    cols_keep = ['timestamp', 'p_short', 'p_wait', 'p_long'] + (['u_short', 'u_wait', 'u_long'] if has_u else [])
    m = pq_map[base_k].rename(columns={
        'p_short': f'ps_{base_k}', 'p_wait': f'pw_{base_k}', 'p_long': f'pl_{base_k}',
        'u_short': f'us_{base_k}', 'u_wait': f'uw_{base_k}', 'u_long': f'ul_{base_k}',
    })
    for k in keys:
        if k == base_k: continue
        dk = pq_map[k][cols_keep].rename(columns={
            'p_short': f'ps_{k}', 'p_wait': f'pw_{k}', 'p_long': f'pl_{k}',
            'u_short': f'us_{k}', 'u_wait': f'uw_{k}', 'u_long': f'ul_{k}',
        })
        m = pd.merge_asof(m, dk, on='timestamp', direction='backward')
    m = m.dropna()
    ens_dict = {
        'timestamp': m['timestamp'], 'close': m['close'],
        'p_short': sum(m[f'ps_{k}'] for k in keys) / len(keys),
        'p_wait':  sum(m[f'pw_{k}'] for k in keys) / len(keys),
        'p_long':  sum(m[f'pl_{k}'] for k in keys) / len(keys),
    }
    for extra_col in ('high', 'low', 'local_vol'):
        if extra_col in m.columns:
            ens_dict[extra_col] = m[extra_col]
    if has_u:
        ens_dict.update({
            'u_short': sum(m[f'us_{k}'] for k in keys) / len(keys),
            'u_wait':  sum(m[f'uw_{k}'] for k in keys) / len(keys),
            'u_long':  sum(m[f'ul_{k}'] for k in keys) / len(keys),
        })
    ens_df = pd.DataFrame(ens_dict)
    print('===', len(keys), 'SCALE ENSEMBLE:', keys, '===')
    for hb in (H[-2], H[-1]):
        run_nonoverlapping_backtest(ens_df, horizon_bars=hb, label=f'ENSEMBLE_{len(keys)}X')
    run_triple_barrier_backtest(ens_df, horizon_bars=H[-1], label=f'ENSEMBLE_{len(keys)}X')
else:
    print('Ensemble waiting for >=2 completed profiles. Ready:', list(ready.keys()))